# 9. An Introduction to the Short Run

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quarcs-lab/macro102-site/blob/main/notebooks/ch09-an-introduction-to-the-short-run.ipynb) [![Open Lecture Slides](https://img.shields.io/badge/Open-Lecture_Slides-8e44ad?logo=revealdotjs&logoColor=white)](https://quarcs-lab.github.io/macro102-site/slides/ch09/)

<!-- BEGIN slides (managed by /write-slides) -->
<a class="chapter-mindmap" href="https://raw.githubusercontent.com/quarcs-lab/macro102-site/main/images/mindmaps/ch09-an-introduction-to-the-short-run-v1.png"><img src="https://raw.githubusercontent.com/quarcs-lab/macro102-site/main/images/mindmaps/ch09-an-introduction-to-the-short-run-v1.webp" alt="Mind map of Chapter 9: An Introduction to the Short Run" width="100%"></a>
<!-- END slides -->

## Learning goals

The long-run model of Chapters 3–8 tells us where the economy is heading on average. This chapter
looks at the ups and downs around that path. By the end of it you will be able to:

- split actual output into **potential output** $\bar{Y}_t$ and **short-run output** $\tilde{Y}_t$, derive
  its percent, log and growth-rate forms, and compute $\tilde{Y}_t$ from data;
- describe U.S. recessions with data on output, employment and inflation, and measure what they cost;
- explain the **Phillips curve** and solve it forward in time, to derive the inflation path of a lasting boom
  and the output cost of bringing inflation down;
- use **Okun's law** to move between short-run output and unemployment, derive its growth-rate form, and read
  the comparative statics of both relationships (sign and slope of each effect, and which line moves);
- in Python: shade recessions on a chart, fit a straight line with `np.polyfit`, compare two vintages
  of the same data, and simulate inflation year by year with a loop.

## Setup

We first load the Python libraries we need. You only have to run this cell once.

> **Python note.** `import` loads a *library*, a toolbox written by other people. We give each
> library a short nickname, for example `np` for NumPy, so that the code stays short. Text after
> a `#` is a *comment*: Python ignores it, and it is there for human readers. The `try ... except`
> block runs one Colab-only line and simply skips it anywhere else.

In [1]:
# Libraries used in this chapter
import numpy as np                              # numbers and arrays
import pandas as pd                             # data tables
import plotly.express as px                     # quick charts from data tables
import plotly.graph_objects as go               # charts built piece by piece (model figures)
import plotly.io as pio
from plotly.subplots import make_subplots       # several panels in one figure
from ipywidgets import interact, interact_manual, FloatSlider, ToggleButtons, Dropdown, SelectMultiple

# One light blue/green look for every chart
BLUE, TEAL, GREEN, GRAY = "#16528c", "#008070", "#7fb23a", "#8a96a3"
pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.update(width=860, height=480)   # same size in Colab and on the website
px.defaults.color_discrete_sequence = [BLUE, TEAL, GREEN, "#5aa0d8", "#0b4f47"]
WIDE = {"description_width": "initial"}         # show the full slider labels

# Colab only: let Plotly charts appear inside the interactive widgets
try:
    import google.colab
    google.colab.output.enable_custom_widget_manager()
except ImportError:
    pass                                        # not running in Colab: nothing to do

# The data files of this chapter live in the book's GitHub repository
DATA_URL = "https://raw.githubusercontent.com/quarcs-lab/macro102-site/main/data/ch09/"

The next cell defines a helper that only the book website needs. Run it and move on.

In [2]:
# @title Helper for the website versions (run this cell; you do not need to read it)
def slider_figure(make_figure, name, values, start, label=None):
    """Precompute make_figure(name=value) for every value and join the results with one slider.

    The website cannot run Python, so each interactive figure also gets this precomputed version.
    """
    figures = [make_figure(**{name: v}) for v in values]
    first = values.index(start)
    fig = go.Figure(layout=figures[first].layout)
    owner = []                                  # which precomputed figure each trace comes from
    for i, f in enumerate(figures):
        for trace in f.data:
            trace.visible = (i == first)
            fig.add_trace(trace)
            owner.append(i)
    steps = []
    for i, f in enumerate(figures):
        layout = {"title": f.layout.title.to_plotly_json(),
                  "annotations": [a.to_plotly_json() for a in f.layout.annotations],
                  "shapes": [s.to_plotly_json() for s in f.layout.shapes]}
        visible = [o == i for o in owner]
        steps.append(dict(method="update", label=str(values[i]), args=[{"visible": visible}, layout]))
    fig.update_layout(sliders=[dict(active=first, steps=steps, pad=dict(t=60),
                                    currentvalue=dict(prefix=(label or name) + " = "))])
    return fig

In [3]:
# @title Helper for the website's country legends (run this cell; you do not need to read it)
def add_legend_traces(fig, traces, keep=()):
    """Website: add traces that start hidden but stay in the legend (click to show), except those named in keep.

    If the figure has a slider, its steps only switch the figure's own traces, so the countries you
    clicked on stay on the chart when you move the slider.
    """
    n = len(fig.data)
    for t in traces:
        if t.name not in keep:
            t.visible = "legendonly"
        fig.add_trace(t)
    for slider in fig.layout.sliders:
        steps = []
        for step in slider.steps:
            steps.append(dict(method=step.method, label=step.label, args=[step.args[0], step.args[1], list(range(n))]))
        slider.steps = steps
    return fig

## Model overview

From one year to the next, the economy does not glide along its long-run path. It overheats and it slumps, workers lose jobs and find new ones, and inflation drifts up and down. This chapter does not yet build a complete model of these movements. Instead it sets up the **frameworks** that the next chapters build on, and they answer three questions. How do we split output into a trend and a cycle? What does the cycle do to inflation? And what does it do to unemployment?

The road map follows those questions. The first block defines potential output and short-run output, and shows how to work with them as percentages, as log differences and as growth rates. The second block writes down the **Phillips curve** and solves it forward in time, which gives the inflation cost of a boom and the output cost of bringing inflation down. The third block adds **Okun's law**, which translates an output gap into an unemployment gap. The fourth block previews how the full short-run model of Chapters 11–13 connects these pieces. At the end, `### The model in code` turns the key results into Python functions and checks each of them with numbers.

**Notation.** A bar marks a value that the short-run model takes as given: potential output $\bar{Y}_t$, the long-run inflation rate $\bar{\pi}$ and the natural rate of unemployment $\bar{u}$ come from the long-run model, and parameters such as the slope $\bar{\nu}$ (the Greek letter *nu*) are chosen from outside. A tilde marks a short-run deviation from the long run, as in $\tilde{Y}_t$. The symbol $\Delta$ is the change from the year before, for example $\Delta\pi_t \equiv \pi_t - \pi_{t-1}$. Short-run output is measured in percent and inflation in percent, so changes in inflation are in percentage points ("points"), as in the textbook's figures.

**One baseline for every diagram.** The textbook's Figs. 9.1 and 9.6 print no numbers, so every model figure and every worked example below starts from one illustrative baseline:

- potential output starts at $\bar{Y}_0 = 100$ and grows by $\bar{g} = 2$ percent a year;
- short-run output follows a wave between $+2$ and $-2$ percent (the $\pm 2\%$ marked in Fig. 9.1) that repeats every 8 years, $\tilde{Y}_t = 2\sin(2\pi t/8)$ percent. In whole years it takes the values $0, +1.41, +2, +1.41, 0, -1.41, -2, -1.41, 0$;
- the slope of the Phillips curve is $\bar{\nu} = 0.33$, the textbook's "about 1/3" for the United States;
- long-run inflation is $\bar{\pi} = 2$ percent, and Okun's slope is $-1/2$.

Fig. 9 is the worked answer to a textbook exercise, which gives its own numbers ($\bar{\nu} = 1$, $\tilde{Y} = +3\%$), and we use those there.

### Trend and cycle: potential output and short-run output

#### Definitions

**Two models with two jobs.** The book uses two models side by side:

| Model | What it determines | Built in |
|---|---|---|
| long-run model | potential output $\bar{Y}_t$ and long-run inflation $\bar{\pi}$ | Chapters 3–8 (the combined Solow–Romer model of [Chapter 6](https://quarcs-lab.github.io/macro102-site/notebooks/ch06-growth-and-ideas.html), the quantity theory of Chapter 8) |
| short-run model | actual output $Y_t$ and actual inflation $\pi_t$, year by year or quarter by quarter | Chapters 9–15 |

In the short-run model, $\bar{Y}_t$ and $\bar{\pi}$ are **exogenous**: they are handed over from the long-run model, which is why they wear bars. The **endogenous** variables are $Y_t$ and $\pi_t$. The long-run inflation rate has no time subscript, because in the end it is a stable target that policymakers pick (Chapter 13).

**Potential output** $\bar{Y}_t$ is what the economy produces when capital, labor and ideas are used at rates it can keep up year after year. It is a sustainable level, not a ceiling: for a while, firms can produce more than potential by running overtime shifts or by putting off maintenance, and they can also produce less.

**Shocks** push actual output away from potential: a jump in oil prices, a freeze in financial markets, a tax cut, a new technology that delivers or disappoints, a natural disaster, a boom or slump abroad. **The short run** is the stretch of time over which such deviations last, typically about two years.

**Short-run output.** Any output level can be split into the trend and the deviation from it, $Y_t = \bar{Y}_t + (Y_t - \bar{Y}_t)$. A gap of 100 million dollars is large for a small economy and tiny for a big one, so we scale the deviation by potential output:

$$\tilde{Y}_t \equiv \frac{Y_t - \bar{Y}_t}{\bar{Y}_t}. \tag{1}$$

$\tilde{Y}_t$ is the **percentage gap** between actual and potential output. It is also called the **output gap** or **detrended output**, because it is what is left of output once the trend has been taken out. In a boom $Y_t > \bar{Y}_t$ and $\tilde{Y}_t > 0$; in a slump $\tilde{Y}_t < 0$; at potential $\tilde{Y}_t = 0$.

**Recessions.** A **recession** starts when short-run output turns negative, and it is usually dated as over once short-run output reaches its low point and starts to climb, even though output is still below potential. In practice the U.S. dates of each peak and trough are set by a committee of the National Bureau of Economic Research (NBER), which looks at employment, sales and industrial production as well as GDP.

#### Working with the definitions

**Step 1: actual output as potential output times a factor.** Multiply both sides of equation (1) by $\bar{Y}_t$, then add $\bar{Y}_t$ to both sides:

$$\tilde{Y}_t\bar{Y}_t = Y_t - \bar{Y}_t \quad\Longrightarrow\quad Y_t = \bar{Y}_t + \tilde{Y}_t\bar{Y}_t,$$

and factor out $\bar{Y}_t$:

$$Y_t = \bar{Y}_t\,(1 + \tilde{Y}_t). \tag{2}$$

Actual output is the trend multiplied by a cycle factor that is a little above 1 in a boom and a little below 1 in a slump. This is exactly how the function `actual_output` builds the baseline: the trend $\bar{Y}_t = 100\times(1.02)^t$ (compound growth, [Appendix C.3](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.3-exponents-and-exponential-growth)) times the factor $1 + 0.02\sin(2\pi t/8)$. In year 2, at the top of the boom, potential output is $100\times1.02^2 = 104.04$ and $\tilde{Y}_2 = +2\%$, so actual output is $104.04\times1.02 = 106.12$. In year 6, at the bottom of the slump, potential is $100\times1.02^6 = 112.62$ and actual output is $112.62\times0.98 = 110.36$. Going back with equation (1): $(110.36 - 112.62)/112.62 = -0.02$, that is, $-2$ percent.

**Step 2: the trend drops out.** In equation (1), the numerator and the denominator both contain the trend. In the baseline, $Y_t - \bar{Y}_t = \bar{Y}_t\times0.02\sin(2\pi t/8)$, so dividing by $\bar{Y}_t$ cancels the growth factor $(1.02)^t$ completely and leaves $\tilde{Y}_t = 2\sin(2\pi t/8)$ percent, whatever the growth rate of potential output. Faster trend growth makes the cycle harder to *see* in a chart of $Y_t$, but it does not change $\tilde{Y}_t$ (Fig. 1).

**Step 3: short-run output as a log difference.** Take the natural log of equation (2). The log of a product is the sum of the logs ([Appendix C.4](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.4-logarithms)):

$$\ln Y_t = \ln\bar{Y}_t + \ln(1 + \tilde{Y}_t).$$

For a small number $x$, $\ln(1+x) \approx x$ ([Appendix C.6](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.6-approximations)). Using it with $x = \tilde{Y}_t$ and moving $\ln\bar{Y}_t$ to the left:

$$\tilde{Y}_t \approx \ln Y_t - \ln\bar{Y}_t. \tag{3}$$

The gap in percent is, to a close approximation, the difference between the logs of actual and potential output. The approximation is very good for the usual gaps: $\ln(1.02) = 0.0198$ and $\ln(0.98) = -0.0202$, both within 0.03 points of $\pm 2\%$. It gets worse for very deep slumps: a gap of $-10\%$ has a log difference of $\ln(0.90) = -0.105$, half a point off. Many statistical agencies report gaps in logs for exactly this reason: logs turn ratios into differences.

**Step 4: growth rates.** Write equation (3) for year $t$ and for year $t-1$, and subtract the second from the first:

$$\ln Y_t - \ln Y_{t-1} \approx \left(\ln\bar{Y}_t - \ln\bar{Y}_{t-1}\right) + \left(\tilde{Y}_t - \tilde{Y}_{t-1}\right).$$

The change in the log of a variable is approximately its growth rate ([Appendix C.5](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.5-growth-rates-and-the-growth-rules)). With potential output growing at the rate $\bar{g}$:

$$g_{Y,t} \approx \bar{g} + \Delta\tilde{Y}_t. \tag{4}$$

The exact version comes from dividing equation (2) for year $t$ by equation (2) for year $t-1$: $Y_t/Y_{t-1} = (1+\bar{g})\times(1+\tilde{Y}_t)/(1+\tilde{Y}_{t-1})$.

*Intuition.* Actual growth is trend growth plus the change in the gap. When the gap widens upward, output grows faster than its trend; when the gap shrinks or turns more negative, it grows more slowly. In the baseline, from year 1 to year 2 the gap rises from $+1.41$ to $+2$, so output grows by about $2 + 0.59 = 2.59$ percent. From year 4 to year 5 the gap falls from $0$ to $-1.41$, so output grows by only about $2 - 1.41 = 0.59$ percent: still growing, yet sliding into a slump.

**Step 5: quarterly and annualized rates.** GDP is published every quarter, and quarterly growth rates are usually reported as **annualized rates**: the growth that would result if the quarter's pace lasted four quarters. Compounding gives $(1+g_q)^4 - 1$, which is close to $4g_q$ when $g_q$ is small. A quarter with 1 percent growth is reported as $1.01^4 - 1 = 4.06\%$, about 4 percent. Going the other way, potential output growing by 2 percent a year grows by $1.02^{1/4} - 1 = 0.496\%$ per quarter, just under a quarter of 2 percent.

#### What the framework predicts

**Growth alone does not reveal the state of the economy.** Equation (4) has one number we observe, $g_{Y,t}$, and two we do not, $\bar{g}$ and $\Delta\tilde{Y}_t$. Suppose GDP grows at an annualized 5 percent. That fits potential growth of 5 percent with an unchanged gap, but it equally fits potential growth of 2 percent with a gap that widened by 3 points. Potential output is never observed directly, so it must be estimated. The common methods are: fit a smooth trend through real GDP; average the actual GDP of the surrounding quarters; or, like the U.S. Congressional Budget Office (CBO), compute what a Solow-style production function gives when labor and capital are at their sustainable levels. Each method can be revised as new data arrive, and the measured gaps change with it (Fig. 3).

**A slump does not require falling output.** In the baseline, actual output never shrinks: its slowest growth is about 0.6 percent a year. Yet the economy spends four years of every eight below potential. A recession is a matter of the gap, not of the sign of growth.

**The cost of a slump.** Add up short-run output over the years below potential. The sum is the output lost, as a percent of one year's potential output. In the baseline the slump years 5, 6 and 7 give $-1.41 - 2 - 1.41 = -4.83$ percent of a year's potential output. Over the whole cycle, though, the boom years add $+4.83$, so the total is exactly zero. Removing fluctuations altogether would therefore gain much less than the cost of a recession on its own, because the booms would go too.

**Comparative statics.**

| Change | Path of $\bar{Y}_t$ | Short-run output $\tilde{Y}_t$ | Growth of actual output $g_Y$ |
|---|---|---|---|
| faster potential growth $\bar{g}$ | steeper trend | unchanged (step 2) | rises one for one, equation (4) |
| bigger swings of the cycle | unchanged | larger booms and slumps | larger swings around $\bar{g}$ |
| larger economy, higher $\bar{Y}_0$ | scaled up | unchanged | unchanged |

In logs, equation (3) reads $\ln Y_t \approx \ln\bar{Y}_t + \tilde{Y}_t$: the elasticity of actual output with respect to potential output is 1, and one point of short-run output adds about one percent to actual output.

### The Phillips curve

#### Assumptions

The short-run model rests on three premises. We state them in our own words:

1. **Shocks never stop.** Something is always hitting the economy, pushing $Y_t$ away from $\bar{Y}_t$ or inflation away from $\bar{\pi}$.
2. **Policy can move output in the short run.** Monetary and fiscal policy can raise or lower spending, and so output, for a while. In this chapter we go one step further and let policymakers choose $\tilde{Y}_t$ directly. Chapters 11 and 12 explain how they do it.
3. **Output and inflation trade off over time.** A hot economy speeds up inflation and a cold one slows it down. This is the Phillips curve:

$$\Delta\pi_t \equiv \pi_t - \pi_{t-1} = \bar{\nu}\,\tilde{Y}_t, \qquad \bar{\nu} > 0. \tag{5}$$

The slope $\bar{\nu}$ is the number of points by which inflation rises in a year for each percent that output is above potential.

*Why the curve slopes up.* When demand runs hot, firms are working beyond their sustainable level: overtime is paid at premium wages, machines run without a break, and goods are scarce. Each firm then raises its prices by more than the inflation everyone expected, and when all firms do so at once, inflation itself goes up. When demand is weak, firms with idle capacity hold back their price increases to win customers, and inflation comes down. The curve describes the *change* in inflation, not its level: in Chapter 12 a full version adds a term for price shocks such as oil.

#### Solving the model

**Step 1: a rule that moves inflation forward.** Add $\pi_{t-1}$ to both sides of equation (5):

$$\pi_t = \pi_{t-1} + \bar{\nu}\,\tilde{Y}_t.$$

This is a **difference equation** ([Appendix C.10](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.10-difference-equations-how-macro-variables-move-over-time)): given last year's inflation and this year's short-run output, it gives this year's inflation.

**Step 2: apply it year after year.** Start from inflation $\pi_0$ in year 0:

$$\pi_1 = \pi_0 + \bar{\nu}\tilde{Y}_1,$$
$$\pi_2 = \pi_1 + \bar{\nu}\tilde{Y}_2 = \pi_0 + \bar{\nu}\left(\tilde{Y}_1 + \tilde{Y}_2\right),$$
$$\pi_3 = \pi_2 + \bar{\nu}\tilde{Y}_3 = \pi_0 + \bar{\nu}\left(\tilde{Y}_1 + \tilde{Y}_2 + \tilde{Y}_3\right),$$

and after $t$ years

$$\pi_t = \pi_0 + \bar{\nu}\sum_{s=1}^{t}\tilde{Y}_s. \tag{6}$$

*Intuition.* Inflation keeps a running total of every past gap. A boom that is over still leaves inflation higher, until a slump of the same total size brings it back.

**Step 3: steady inflation.** Inflation stays constant, $\Delta\pi_t = 0$, if and only if $\tilde{Y}_t = 0$. Any inflation rate can persist as long as output sits at potential. That is why the level of long-run inflation $\bar{\pi}$ is not pinned down by the Phillips curve: it comes from the long run and from policy.

**Step 4: a boom that lasts.** If policymakers hold output at the same gap $\tilde{Y}$ every year, the sum in equation (6) is $t$ copies of $\tilde{Y}$, so $\pi_t = \pi_0 + \bar{\nu}\tilde{Y}\,t$: inflation climbs in a straight line, by $\bar{\nu}\tilde{Y}$ points every year, without end. With the baseline slope $0.33$ and a gap held at $+2$ percent, inflation rises by $0.66$ points a year: $2 \to 2.66 \to 3.32 \to 3.98$ percent. Fig. 9 runs this experiment.

**Step 5: the output cost of lowering inflation.** Set $t = T$ in equation (6), subtract $\pi_0$ from both sides and divide by $\bar{\nu}$:

$$\sum_{s=1}^{T}\tilde{Y}_s = \frac{\pi_T - \pi_0}{\bar{\nu}}. \tag{7}$$

To lower inflation, the gaps must add up to a negative number, and the required total is fixed by the slope alone. With $\bar{\nu} = 0.33$, cutting inflation by one point takes a cumulative gap of $-1/0.33 = -3.03$ percent-years: for example $-3.03$ percent for one year, or $-1.01$ percent in each of three years. Policy chooses how to spread the pain over time, not how much pain there is in total. The ratio $1/\bar{\nu}$, the output lost per point of disinflation, is often called the **sacrifice ratio**.

#### What the model predicts

| Short-run output | Change in inflation | Path of inflation |
|---|---|---|
| $\tilde{Y}_t > 0$ (boom) | $\Delta\pi_t = \bar{\nu}\tilde{Y}_t > 0$ | rising |
| $\tilde{Y}_t = 0$ (at potential) | $\Delta\pi_t = 0$ | steady at any level |
| $\tilde{Y}_t < 0$ (slump) | $\Delta\pi_t = \bar{\nu}\tilde{Y}_t < 0$ | falling |

**Comparative statics.** The effect of one more percent of short-run output on the change in inflation is the slope, $\partial\Delta\pi_t/\partial\tilde{Y}_t = \bar{\nu}$. Both variables are already in percent, so the slope plays the role of an elasticity.

- A **steeper curve** (higher $\bar{\nu}$) rotates the line counterclockwise around the origin. A boom then raises inflation faster, and a disinflation is cheaper: by equation (7), its output cost $1/\bar{\nu}$ falls.
- A **price shock** (Chapter 12) shifts the whole line up: inflation rises even at potential.
- A different long-run inflation rate $\bar{\pi}$ does not move the line at all, because the curve is about changes in inflation.

**Inflation peaks as the slump begins.** By equation (6), inflation rises while $\tilde{Y}_t > 0$ and falls while $\tilde{Y}_t < 0$. Its highest point is therefore the moment the gap crosses zero from above, which is the start of a recession. In the baseline cycle, inflation goes $2 \to 2.47 \to 3.13 \to 3.59$ percent in years 1 to 3, stays at $3.59$ in year 4 (where $\tilde{Y} = 0$), and then falls back to 2 percent by year 8, because the gaps of a full cycle add up to zero. This is the pattern of Fig. 5: U.S. inflation tends to peak close to the start of each recession and to fall during it.

**No free lunch from a permanent boom.** Holding output above potential buys ever-rising inflation (step 4). Lowering inflation that is too high requires a period of output below potential (equation 7), which is how the Volcker disinflation of the early 1980s worked (Fig. 6).

### Okun's law

#### Definitions

The **unemployment rate** is $u_t$. The **natural rate of unemployment** $\bar{u}$ is the rate the economy settles at in the long run ([Chapter 7](https://quarcs-lab.github.io/macro102-site/notebooks/ch07-the-labor-market-wages-and-unemployment.html)), the labor-market cousin of potential output. Their difference $u_t - \bar{u}$ is **cyclical unemployment**. Okun's law is a rule of thumb fitted to U.S. data (Fig. 8):

$$u_t - \bar{u} = -\tfrac{1}{2}\,\tilde{Y}_t. \tag{8}$$

Each percent of output below potential comes with half a point of unemployment above the natural rate. The slope is smaller than 1 in size because firms do not adjust only through jobs: in a slump they also cut hours and keep some workers they do not fully need, and some people who lose jobs stop looking for work and so are not counted as unemployed.

#### Working with the law

**Step 1: going back from unemployment to output.** Multiply both sides of equation (8) by $-2$:

$$\tilde{Y}_t = -2\,(u_t - \bar{u}).$$

In the baseline slump, $\tilde{Y} = -2\%$ means unemployment 1 point above its natural rate, and 1 point of unemployment below the natural rate signals output 2 percent above potential.

**Step 2: changes in unemployment.** Write equation (8) for years $t$ and $t-1$ and subtract. The natural rate is constant, so it cancels:

$$\Delta u_t = -\tfrac{1}{2}\,\Delta\tilde{Y}_t.$$

Now replace $\Delta\tilde{Y}_t$ with $g_{Y,t} - \bar{g}$ from equation (4):

$$\Delta u_t \approx -\tfrac{1}{2}\left(g_{Y,t} - \bar{g}\right). \tag{9}$$

*Intuition.* Unemployment rises whenever output grows more slowly than potential, even if output is still growing. Steady growth at the trend rate only holds unemployment where it is. In the baseline, from year 4 to year 5 output grows only about 0.6 percent, so unemployment rises by about $-\tfrac{1}{2}(0.6 - 2) = +0.7$ point.

**Step 3: the Phillips curve in terms of unemployment.** Substitute $\tilde{Y}_t = -2(u_t - \bar{u})$ from step 1 into the Phillips curve, equation (5):

$$\Delta\pi_t = \bar{\nu}\times\left[-2\,(u_t - \bar{u})\right] = -2\bar{\nu}\,(u_t - \bar{u}).$$

Inflation rises when unemployment is below its natural rate and falls when it is above. With $\bar{\nu} = 0.33$, each point of unemployment above the natural rate lowers inflation by $0.66$ points a year. This is the form in which the trade-off was first found in the data, with unemployment on the horizontal axis (Chapter 12 tells that history).

#### What the law predicts

| Short-run output $\tilde{Y}_t$ | $+2\%$ | $0$ | $-2\%$ | $-4\%$ |
|---|---|---|---|---|
| Cyclical unemployment $u_t - \bar{u}$ (points) | $-1$ | $0$ | $+1$ | $+2$ |

- Booms push unemployment below the natural rate, slumps push it above, and the two move together closely, as the tight cloud of Fig. 8 shows.
- A steeper Okun slope (larger in size) would mean bigger swings in unemployment for the same output gap.
- Combined with equation (7): cutting inflation by one point with $\bar{\nu} = 0.33$ costs $3.03$ percent-years of output, or $\tfrac{1}{2}\times 3.03 \approx 1.5$ point-years of extra unemployment.
- Because the two gaps move together, the rest of the short-run chapters can work with output alone and translate to unemployment with equation (8) whenever needed.

### A preview of the full short-run model

#### Assumptions

The full model keeps the long run as given ($\bar{Y}_t$, $\bar{\pi}$ and $\bar{u}$ come from outside) and keeps the three premises above. It adds households, firms that invest, a government that spends, trade with the rest of the world and a central bank that runs monetary policy.

#### How the pieces fit together

The chain of cause and effect, built over the next chapters, runs from the interest rate to output and then to inflation and unemployment:

$$\underbrace{R_t}_{\text{real interest rate}} \;\xrightarrow{\ \text{IS curve}\ }\; \tilde{Y}_t \;\xrightarrow{\ \text{Phillips curve}\ }\; \Delta\pi_t, \qquad \tilde{Y}_t \;\xrightarrow{\ \text{Okun's law}\ }\; u_t - \bar{u}.$$

- The **IS curve** ([Chapter 11](https://quarcs-lab.github.io/macro102-site/notebooks/ch11-the-is-curve.html)) says that a higher real interest rate $R_t$ lowers investment and other borrowing-financed spending, and so lowers short-run output.
- The **MP curve** (Chapter 12) shows how the central bank, by setting a short-term nominal interest rate such as the federal funds rate, sets the real interest rate in the short run.
- The **Phillips curve**, equation (5), then turns short-run output into a change in inflation, and Okun's law, equation (8), into unemployment.

So the signs run: a higher $R_t$ means a lower $\tilde{Y}_t$, falling inflation and higher unemployment. This chapter skips the first link by letting policymakers pick $\tilde{Y}_t$ directly.

#### What the model predicts

- Policy can lean against shocks and keep $\tilde{Y}_t$ near zero, which keeps inflation steady (equation 5).
- It cannot keep output above potential year after year without inflation climbing without limit (equation 6).
- High inflation comes down only through a period of output below potential, with a total cost set by the slope (equation 7).
- In practice the job is harder than in the model, because potential output is not observed (equation 4) and new shocks keep arriving.

### The model in code

We now turn the key results into Python functions. Each function's description names the equation it computes. The cells after them check the algebra above with numbers.

> **Python note.** `def` creates a *function*: a named recipe that takes inputs (the
> *arguments*) and gives back a result with `return`. Writing `g=2.0` gives the argument a
> *default value*, used whenever we do not pass one. The text in triple quotes is a short
> description of what the function does. `**` means "to the power of", and `np.sin` is the sine
> function, which draws a smooth wave (`np.pi` is the number π).

In [4]:
def potential_output(t, g=2.0, Y0=100.0):
    """Potential output Ybar_t: a smooth trend that grows g percent a year, from Y0 in year 0."""
    return Y0 * (1 + g / 100) ** t


def actual_output(t, A=2.0, g=2.0, cycle=8):
    """Actual output Y_t = Ybar_t (1 + Y~_t)   (equation 2), with a wave of +-A percent that repeats every `cycle` years."""
    return potential_output(t, g) * (1 + A / 100 * np.sin(2 * np.pi * t / cycle))


def short_run_output(Y, Ybar):
    """Short-run output in percent: 100 (Y - Ybar) / Ybar   (equation 1)."""
    return 100 * (Y - Ybar) / Ybar

In [5]:
def phillips_curve(Y_tilde, nu=0.33):
    """Change in inflation, percentage points: delta pi = nu * Y_tilde   (the Phillips curve, equation 5)."""
    return nu * Y_tilde


def okun_law(Y_tilde, slope=-0.5):
    """Cyclical unemployment u - u_bar, percentage points: slope * Y_tilde   (Okun's law, equation 8)."""
    return slope * Y_tilde

**Check 1: equations (1) and (2), and the trend drops out.** At the top of the boom (year 2) and the bottom of the
slump (year 6), we compute short-run output from the baseline and rebuild actual output from it. Then we change
the growth of potential output and recompute short-run output in year 6.

> **Python note.** An *f-string* such as `f"Y~ = {value:+.2f}%"` inserts the value of a variable
> into a piece of text; `:.2f` means "show two decimals" and the `+` adds a sign. A `for` loop repeats its
> indented lines once for each value in the list after `in`. `Ybar, Y = a, b` gives two variables their values
> in one line.

In [6]:
for t in (2, 6):
    Ybar, Y = potential_output(t), actual_output(t)
    Y_tilde = short_run_output(Y, Ybar)
    print(f"Year {t}: potential {Ybar:.2f}, actual {Y:.2f}, Y~ = {Y_tilde:+.2f}% (equation 1)")
    print(f"        equation (2): Ybar x (1 + Y~) = {Ybar * (1 + Y_tilde / 100):.2f}")
for g in (0.0, 2.0, 4.0):
    Y_tilde = short_run_output(actual_output(6, g=g), potential_output(6, g=g))
    print(f"Potential grows {g:.0f}% a year: Y~ in year 6 = {Y_tilde:+.2f}%")

Year 2: potential 104.04, actual 106.12, Y~ = +2.00% (equation 1)
        equation (2): Ybar x (1 + Y~) = 106.12
Year 6: potential 112.62, actual 110.36, Y~ = -2.00% (equation 1)
        equation (2): Ybar x (1 + Y~) = 110.36
Potential grows 0% a year: Y~ in year 6 = -2.00%
Potential grows 2% a year: Y~ in year 6 = -2.00%
Potential grows 4% a year: Y~ in year 6 = -2.00%


**Check 2: the log difference, equation (3).** For gaps of different sizes we compare the exact gap with
$100\times(\ln Y - \ln\bar{Y}) = 100\ln(1+\tilde{Y})$.

> **Python note.** `np.log` is the natural logarithm. `:+6.1f` reserves 6 characters for the number, so that
> the columns line up.

In [7]:
for gap in (2.0, -2.0, -5.0, -10.0):
    log_gap = 100 * np.log(1 + gap / 100)                     # ln Y - ln Ybar, in percent
    print(f"Y~ = {gap:+6.1f}%   log difference = {log_gap:+7.2f}%   error {log_gap - gap:+.2f} points")

Y~ =   +2.0%   log difference =   +1.98%   error -0.02 points
Y~ =   -2.0%   log difference =   -2.02%   error -0.02 points
Y~ =   -5.0%   log difference =   -5.13%   error -0.13 points
Y~ =  -10.0%   log difference =  -10.54%   error -0.54 points


**Check 3: growth rates, equation (4).** Year by year over one cycle, we compare the exact growth rate of actual
output with potential growth (2 percent) plus the change in short-run output.

> **Python note.** `range(1, 9)` counts 1, 2, ..., 8, so the loop visits every year of one cycle.

In [8]:
for t in range(1, 9):
    exact = 100 * (actual_output(t) / actual_output(t - 1) - 1)
    gap_now = short_run_output(actual_output(t), potential_output(t))
    gap_before = short_run_output(actual_output(t - 1), potential_output(t - 1))
    approx = 2.0 + (gap_now - gap_before)                     # equation (4) with g = 2
    print(f"Year {t}: Y~ = {gap_now:+5.2f}%   growth of Y = {exact:4.2f}% (exact), {approx:4.2f}% (equation 4)")

Year 1: Y~ = +1.41%   growth of Y = 3.44% (exact), 3.41% (equation 4)
Year 2: Y~ = +2.00%   growth of Y = 2.59% (exact), 2.59% (equation 4)
Year 3: Y~ = +1.41%   growth of Y = 1.41% (exact), 1.41% (equation 4)
Year 4: Y~ = +0.00%   growth of Y = 0.58% (exact), 0.59% (equation 4)
Year 5: Y~ = -1.41%   growth of Y = 0.56% (exact), 0.59% (equation 4)
Year 6: Y~ = -2.00%   growth of Y = 1.39% (exact), 1.41% (equation 4)
Year 7: Y~ = -1.41%   growth of Y = 2.61% (exact), 2.59% (equation 4)
Year 8: Y~ = +0.00%   growth of Y = 3.46% (exact), 3.41% (equation 4)


**Check 4: the Phillips curve over one cycle, equations (5) and (6).** We move inflation forward one year at a
time with equation (5), and compare it with the running-total formula, equation (6). Inflation should peak in
years 3 and 4 and be back at 2 percent by the end of the cycle.

> **Python note.** `pi = pi + ...` replaces last year's inflation with this year's. `total_gap` starts at zero
> and adds one year's gap each time round the loop: a *running total*.

In [9]:
pi = 2.0                                                      # year 0: inflation at its long-run rate
total_gap = 0.0                                               # running total of short-run output
for t in range(1, 9):
    Y_tilde = short_run_output(actual_output(t), potential_output(t))
    pi = pi + phillips_curve(Y_tilde)                         # one year of equation (5)
    total_gap = total_gap + Y_tilde
    formula = 2.0 + 0.33 * total_gap                          # equation (6)
    print(f"Year {t}: Y~ = {Y_tilde:+5.2f}%   inflation {pi:.3f}% (year by year), {formula:.3f}% (equation 6)")

Year 1: Y~ = +1.41%   inflation 2.467% (year by year), 2.467% (equation 6)
Year 2: Y~ = +2.00%   inflation 3.127% (year by year), 3.127% (equation 6)
Year 3: Y~ = +1.41%   inflation 3.593% (year by year), 3.593% (equation 6)
Year 4: Y~ = +0.00%   inflation 3.593% (year by year), 3.593% (equation 6)
Year 5: Y~ = -1.41%   inflation 3.127% (year by year), 3.127% (equation 6)
Year 6: Y~ = -2.00%   inflation 2.467% (year by year), 2.467% (equation 6)
Year 7: Y~ = -1.41%   inflation 2.000% (year by year), 2.000% (equation 6)
Year 8: Y~ = +0.00%   inflation 2.000% (year by year), 2.000% (equation 6)


The same running total measures the **cost of the slump**: the gaps of years 5 to 7 add up to the output lost, and
the whole cycle adds up to zero.

> **Python note.** `abs` is the absolute value. We use it on the full-cycle total only to drop a meaningless minus
> sign: the computer's answer is a tiny number such as $-0.0000000000000004$.

In [10]:
lost = 0.0
for t in (5, 6, 7):
    lost = lost + short_run_output(actual_output(t), potential_output(t))
print(f"Output lost in years 5-7: {lost:.2f} percent of a year's potential output")
print(f"Sum of Y~ over the whole cycle (years 1-8): {abs(total_gap):.2f}")

Output lost in years 5-7: -4.83 percent of a year's potential output
Sum of Y~ over the whole cycle (years 1-8): 0.00


**Check 5: the cost of a disinflation, equation (7).** To lower inflation from 2 to 1 percent with
$\bar{\nu} = 0.33$, equation (7) asks for a cumulative gap of $-3.03$ percent-years. We try it all in one year and
spread over three years: both must end at 1 percent.

> **Python note.** `[a, b, c]` is a *list*. A loop can sit inside another loop: the outer one picks a path, the
> inner one runs through its years. `len(path)` is the number of years in the path.

In [11]:
needed = (1.0 - 2.0) / 0.33                                   # equation (7)
print(f"Cumulative Y~ needed: {needed:.2f} percent-years")
fast = [needed]                                               # all of it in one year
slow = [needed / 3, needed / 3, needed / 3]                   # spread over three years
for path in (fast, slow):
    pi = 2.0
    for Y_tilde in path:
        pi = pi + phillips_curve(Y_tilde)
    print(f"{len(path)} year(s) at Y~ = {path[0]:+.2f}%: inflation ends at {pi:.2f}%")
print(f"Okun's law: extra unemployment, summed over the years = {okun_law(needed):+.2f} point-years")

Cumulative Y~ needed: -3.03 percent-years
1 year(s) at Y~ = -3.03%: inflation ends at 1.00%
3 year(s) at Y~ = -1.01%: inflation ends at 1.00%
Okun's law: extra unemployment, summed over the years = +1.52 point-years


**Check 6: Okun's law, equations (8) and (9).** From year 4 to year 5 of the baseline, output still grows. We
compute the change in unemployment from equation (8) and from the growth form, equation (9). Then we check that
the Phillips curve in unemployment terms, $\Delta\pi = -2\bar{\nu}(u - \bar{u})$, gives the same answer as
equation (5).

In [12]:
gap_4 = short_run_output(actual_output(4), potential_output(4))
gap_5 = short_run_output(actual_output(5), potential_output(5))
growth = 100 * (actual_output(5) / actual_output(4) - 1)
change_u = okun_law(gap_5) - okun_law(gap_4)                  # from equation (8)
print(f"Year 4 to 5: output grows {growth:.2f}%, unemployment changes by {change_u:+.2f} points (equation 8)")
print(f"Equation (9): -1/2 x ({growth:.2f} - 2) = {-0.5 * (growth - 2.0):+.2f} points")
u_gap = okun_law(-2.0)                                        # the bottom of the baseline slump
print(f"Y~ = -2%: u - u_bar = {u_gap:+.2f} points")
print(f"Phillips curve with unemployment: -2 x 0.33 x ({u_gap:+.2f}) = {-2 * 0.33 * u_gap:+.2f}; "
      f"with output: {phillips_curve(-2.0):+.2f}")

Year 4 to 5: output grows 0.56%, unemployment changes by +0.71 points (equation 8)
Equation (9): -1/2 x (0.56 - 2) = +0.72 points
Y~ = -2%: u - u_bar = +1.00 points
Phillips curve with unemployment: -2 x 0.33 x (+1.00) = -0.66; with output: -0.66


The two ways of computing the change in unemployment differ a little (0.71 against 0.72 points), because
equation (4) is an approximation.

### Drawing the curves

The textbook writes the name of each curve next to it, so our model figures do the same instead of using a
legend. `side` says whether the name sits above (`"bottom"`), beside (`"middle"`) or below (`"top"`) the end point.

> **Python note.** `a if condition else b` is a one-line choice: it gives `a` when the condition is true and `b`
> otherwise. Here it moves the name 4 pixels away from the point, to the right or to the left.

In [13]:
def add_end_label(fig, x, y, text, color, side="middle", row=None, col=None, xanchor="left"):
    """Write a curve's name next to its end point (x, y), in the curve's color."""
    fig.add_annotation(x=x, y=y, text=text, showarrow=False, xanchor=xanchor, yanchor=side,
                       xshift=4 if xanchor == "left" else -4, font=dict(color=color, size=15), row=row, col=col)

### Shading recessions

Several data figures shade the U.S. recessions, so we write the shading once, here. A small file lists the
start (peak) and end (trough) month of every U.S. recession since 1960, as dated by the National Bureau of
Economic Research (NBER).

> **Python note.** `pd.read_csv` reads a data file into a *DataFrame*, a table with named columns, and
> `parse_dates=[...]` turns text such as `"2020-02-01"` into real dates. A `for` loop repeats the indented
> lines once for each recession: `zip(recessions.peak, recessions.trough)` hands over one (peak, trough) pair
> at a time. The `if` skips recessions that ended before the chart starts, and `fig.add_vrect` shades the
> band between two dates.

In [14]:
recessions = pd.read_csv(DATA_URL + "recessions.csv", parse_dates=["peak", "trough"])


def shade_recessions(fig, start="1960-01-01"):
    """Figs. 9.3 and 9.5: shade every NBER recession (peak to trough) that ends after the date start."""
    for peak, trough in zip(recessions.peak, recessions.trough):
        if trough >= pd.Timestamp(start):
            fig.add_vrect(x0=peak, x1=trough, fillcolor=BLUE, opacity=0.15, line_width=0)


recessions.tail(3)

,peak,trough
6,2001-03-01,2001-11-01
7,2007-12-01,2009-06-01
8,2020-02-01,2020-04-01


### Data for other countries

The textbook's data figures show the United States. The same concepts are measured for other countries too, and
this notebook lets you add them. One file collects them, a year per row, for the OECD members and several large
emerging economies:

In [15]:
panel = pd.read_csv(DATA_URL + "short_run_panel.csv")
panel[panel.year == 2020].head()

,iso3,country,year,gdp_real,gdp_potential,output_gap,unemployment,inflation,inflation_source,unemployment_trend
59,ARG,Argentina,2020,1.07698,NaN,NaN,NaN,42.015,WDI,NaN
125,AUS,Australia,2020,1.50124,1.55782,-3.632,6.473,0.847,WDI,5.083
191,AUT,Austria,2020,0.53734,0.56490,-4.877,5.363,1.382,WDI,5.309
257,BEL,Belgium,2020,0.66142,0.69928,-5.413,5.562,0.737,WDI,6.282
313,BRA,Brazil,2020,3.61542,NaN,NaN,13.722,3.212,WDI,10.617


- `gdp_real` and `gdp_potential` are actual and potential real GDP, in trillions of 2021 dollars at purchasing power
  parity (PPP, prices made comparable across countries), and `output_gap` is short-run output $\tilde{Y}$ in percent.
  All three come from the OECD's *Economic Outlook*. The OECD estimates potential output for its members only, from
  1985 at the earliest.
- `inflation` is CPI inflation from the World Bank (`inflation_source` says which database: `WDI`, the World Bank's
  Global Inflation Database `GID`, or the OECD).
- `unemployment` is the unemployment rate, and `unemployment_trend` a smooth trend through it (more in Fig. 8).

The file also has the OECD's own numbers for the United States. The figures use the textbook's U.S. data (the CBO
and the BLS) instead, and Fig. 3 compares the two. So we keep the other countries in `others`, and give every
country a fixed color.

Several countries are often shown together: all the countries with an output gap (Figs. 3, 7 and 8), and the
groups of the website's menu in Fig. 5. No two countries in the same view should share a color, so we list the
menu's groups first.

> **Python note.** A *dictionary* such as `{"Japan": "#16528c"}` maps a name to a value. Here each group's name maps
> to a list of countries.

In [16]:
GROUPS_9 = {"G7": ["Canada", "France", "Germany", "Italy", "Japan", "United Kingdom"],
            "Euro area": ["Austria", "Belgium", "Croatia", "Estonia", "Finland", "France", "Germany", "Greece", "Ireland",
                          "Italy", "Latvia", "Lithuania", "Luxembourg", "Netherlands", "Portugal", "Slovak Republic",
                          "Slovenia", "Spain"],
            "Other Europe": ["Bulgaria", "Czechia", "Denmark", "Hungary", "Iceland", "Norway", "Poland", "Romania",
                             "Sweden", "Switzerland", "Turkey"],
            "Americas and Asia-Pacific": ["Argentina", "Australia", "Brazil", "Canada", "Chile", "China", "Colombia",
                                          "Costa Rica", "India", "Indonesia", "Israel", "Japan", "Mexico", "New Zealand",
                                          "Peru", "South Korea", "Thailand"],
            "Other emerging economies": ["Russia", "Saudi Arabia", "South Africa", "Turkey", "Bulgaria", "Romania"]}

The colors come from a small tool. Its palette has 44 colors: Plotly's 24 dark ones, and the 20 darker ones of its
26 "Alphabet" colors.

> **Python note.** `PALETTE = []` starts an empty list, and `.append` adds one color at a time. In `first_free`,
> `return` inside a loop stops the function at the first color that fits, `used.count(c)` counts how often `c`
> appears in the list `used`, and `reversed(PALETTE)` walks through the palette from its last color.

In [17]:
LIGHT = ["#F7E1A0", "#E2E2E2", "#FBE426", "#16FF32", "#1CFFCE", "#2ED9FF"]   # too pale on white
PALETTE = []
for c in px.colors.qualitative.Dark24:                                       # 24 dark colors ...
    if c != "#2E91E5":
        PALETTE.append(c)
for c in px.colors.qualitative.Alphabet:
    if c not in LIGHT:
        PALETTE.append(c)                                                     # ... and 20 more: 44 in all
PALETTE.append("#2E91E5")                    # a medium blue, too close to the U.S. navy: used last


def first_free(used, taken):
    """The palette color least used in `used` (the group-mates' colors), preferring one no country has `taken` yet."""
    for c in PALETTE:
        if c not in used and c not in taken:
            return c
    best = PALETTE[-1]
    for c in reversed(PALETTE):                  # from the end, far from the colors always on show
        if used.count(c) < used.count(best):
            best = c                             # a big group shares out the colors evenly
    return best

> **Python note.** `group_colors` walks through the countries one by one. `dict(fixed or {})` starts from a copy of
> the fixed colors (an empty dictionary when there are none), `continue` skips a country that already has a color,
> and `used` collects the colors of the countries that are shown together with this one.

In [18]:
def group_colors(names, groups, fixed=None):
    """Give each country a color that no other country in its groups uses, as far as the palette allows.

    names: every country; groups: the lists of countries shown together (menu views); fixed: {country: color}
    for the textbook's countries, so the default view keeps the book's colors."""
    colors = dict(fixed or {})
    for name in names:
        if name in colors:
            continue
        used = []
        for group in groups:
            if name in group:
                for other in group:
                    if other in colors:
                        used.append(colors[other])
        colors[name] = first_free(used, list(colors.values()))
    return colors

The United States keeps the chapter's blue, and every other country gets a color that no country of its groups uses.

In [19]:
others = panel[panel.country != "United States"]
COUNTRIES = ["United States"] + sorted(others.country.unique())
WITH_GAP = sorted(others.dropna(subset=["output_gap"]).country.unique())   # shown together in Figs. 3, 7 and 8
COLORS = group_colors(COUNTRIES, [WITH_GAP] + list(GROUPS_9.values()), {"United States": BLUE})
print(len(COUNTRIES), "economies:", ", ".join(COUNTRIES[:8]), "...")

51 economies: United States, Argentina, Australia, Austria, Belgium, Brazil, Bulgaria, Canada ...


## Trend and cycle

We start with a made-up economy to see the key idea of the chapter: actual output equals potential output plus short-run fluctuations around it.

### Fig. 1: Economic fluctuations and short-run output

**Goal.** See how removing the trend from actual output leaves the ups and downs of the business cycle.

Panel (a) draws actual output $Y_t$ around a smooth potential output $\bar{Y}_t$. Panel (b) draws short-run
output $\tilde{Y}_t$ from equation (1): the same wiggle, with the trend taken out. The gray bars in panel (a)
are the gaps $Y - \bar{Y}$ at the top of the boom and at the bottom of the slump.

> **Python note.** `np.linspace(0, 8, 200)` creates 200 evenly spaced numbers from 0 to 8: the
> years at which we draw the curves. `make_subplots(rows=1, cols=2)` makes a figure with two panels,
> and `row=..., col=...` tells `add_trace` where to draw. `fig.add_hline` draws a horizontal line. We fix
> the axis ranges so that you can *see* the curves move when a parameter changes.

In [20]:
def fluctuations(A=2.0, g=2.0):
    """Fig. 9.1: actual and potential output (a) and short-run output (b), over one 8-year cycle."""
    t = np.linspace(0, 8, 201)                          # time, in years: t[50] = 2 and t[150] = 6
    Y, Ybar = actual_output(t, A, g), potential_output(t, g)
    fig = make_subplots(rows=1, cols=2, subplot_titles=("(a) Actual and potential output",
                                                        "(b) Short-run output, Ỹ (percent)"))
    fig.add_trace(go.Scatter(x=t, y=Y, name="Actual output, Y", line=dict(color=BLUE, width=3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=Ybar, name="Potential output, Ȳ", line=dict(color=GREEN, width=3, dash="dash")),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=[2, 2, None, 6, 6], y=[Ybar[50], Y[50], None, Ybar[150], Y[150]],   # boom and slump
                             name="Gap, Y − Ȳ", mode="lines", line=dict(color=GRAY, width=4)), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=short_run_output(Y, Ybar), name="Short-run output, Ỹ",
                             line=dict(color=TEAL, width=3)), row=1, col=2)
    fig.add_hline(y=0, line=dict(color=GREEN, width=2, dash="dash"), row=1, col=2)      # Ỹ = 0: output at potential
    add_end_label(fig, 8, Y[-1], "Y", BLUE, "bottom", 1, 1)          # Y and Ȳ meet at t = 8: one above,
    add_end_label(fig, 8, Ybar[-1], "Ȳ", GREEN, "top", 1, 1)         # one below
    add_end_label(fig, 8, 0, "Ỹ", TEAL, "bottom", 1, 2)
    fig.add_annotation(x=2, y=A, xref="x2", yref="y2", text="Boom: Ỹ > 0", showarrow=False, yshift=14)
    fig.add_annotation(x=6, y=-A, xref="x2", yref="y2", text="Slump: Ỹ < 0", showarrow=False, yshift=-14)
    fig.update_xaxes(title_text="Time (years)", range=[0, 8])
    fig.update_yaxes(title_text="Output", range=[94, 145], row=1, col=1)
    fig.update_yaxes(range=[-5.2, 5.2], row=1, col=2)
    fig.update_layout(title=f"Boom: Ỹ = +{A:.1f}%   Slump: Ỹ = −{A:.1f}%   |   potential grows {g:.1f}% a year",
                      showlegend=False)
    return fig

Now the interactive version. Move the sliders and compare the two panels.

> **Python note.** `interact` turns each argument of a function into a slider and calls the
> function again every time you move one. The `;` at the end only hides an extra line of text.

In [21]:
interact(fluctuations,
         A=FloatSlider(value=2.0, min=0.5, max=4.0, step=0.5, description="size of the swings A (%)", style=WIDE),
         g=FloatSlider(value=2.0, min=0.0, max=4.0, step=0.5, description="growth of potential g (% a year)",
                       style=WIDE));

In [22]:
# @title Website version: Fig. 1 with a precomputed slider for the growth of potential output
slider_figure(fluctuations, "g", [0.0, 0.5, 1.0, 2.0, 3.0, 4.0], start=2.0, label="growth of potential g (% a year)")

**Source.** Adapted from Fig. 9.1 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- Panel (b) looks just like actual output in panel (a), without the upward trend. That is why
  economists also call $\tilde{Y}$ *detrended output*.
- Raise the growth of potential output to 4 percent. In panel (a) the slump almost disappears: even at the
  bottom of the cycle, actual output still grows by more than 2 percent a year. In panel (b) nothing changes. Strong growth hides the cycle, which
  is exactly what happens with real GDP in Fig. 2.
- Set $\bar{g} = 0$ and the two panels look the same: potential output is then a constant 100, so $Y - \bar{Y}$ and
  $\tilde{Y}$ are the same numbers.

**Your turn.** With the baseline ($A = 2$, $\bar{g} = 2$), compute short-run output in year 6 with
`short_run_output`, `actual_output` and `potential_output`. Then repeat with `g=4.0`. Does the answer change?

<details><summary>Hint</summary>

Pass the same arguments to both functions, for example `actual_output(6, A=2.0, g=4.0)` and
`potential_output(6, g=4.0)`. Look at equation (1): the trend cancels out.

</details>

In [23]:
# Your turn: try it here

## U.S. fluctuations in the data

Four U.S. series show what a recession looks like: GDP against its potential, short-run output, monthly job changes, and inflation.

### Fig. 2: U.S. real GDP, actual and potential

**Goal.** Look at actual and potential GDP in the United States since 1929.

The Congressional Budget Office (CBO) estimates U.S. potential output from 1949 on, with a version of the
Solow model. Our data file adds an estimate for 1929–1948, a very smooth trend through actual GDP.

> **Python note.** `.head()` shows the first five rows of a DataFrame, which is always a good first look at
> new data.

In [24]:
gdp = pd.read_csv(DATA_URL + "fig9-2.csv")
gdp.head()

,year,gdp_actual,gdp_potential
0,1929,1.40584,1.12786
1,1930,1.28623,1.17955
2,1931,1.20384,1.23361
3,1932,1.04856,1.29019
4,1933,1.03560,1.34940


Each row is a year. `gdp_actual` and `gdp_potential` are real GDP in trillions of 2022 dollars. We first
compute short-run output with our function `short_run_output`, which works on whole columns at once.

> **Python note.** `gdp["gap"] = ...` adds a new column, computed row by row from two other columns.
> `gdp[gdp.year <= 1946]` keeps only the rows where the condition is true.

In [25]:
gdp["gap"] = short_run_output(gdp.gdp_actual, gdp.gdp_potential)
gdp[gdp.year <= 1946].round(2)

,year,gdp_actual,gdp_potential,gap
0,1929,1.41,1.13,24.65
1,1930,1.29,1.18,9.04
2,1931,1.20,1.23,-2.41
3,1932,1.05,1.29,-18.73
4,1933,1.04,1.35,-23.25
5,1934,1.15,1.41,-18.69
6,1935,1.25,1.48,-15.35
7,1936,1.41,1.54,-8.64
8,1937,1.48,1.62,-8.18
9,1938,1.43,1.69,-15.12


The Great Depression shows up clearly: output was 23 percent below potential in 1933. During World War II
it was 35 percent above potential in 1944. But look at the first row: output 25 percent *above* potential in 1929.
That is not a real boom. Nobody published potential output for those years, so the early values are only an
estimate. The trend has to start somewhere, and at the edge of the sample it is least reliable. **Always ask
where a number comes from**, and treat the 1929–1948 gaps with care.

Now the chart. On a **ratio scale** (a logarithmic axis), equal distances mean equal percentage changes. You can
also pick another country. Its GDP comes from the OECD, in trillions of 2021 dollars at PPP, and starts later.

> **Python note.** `rename(columns={...})` gives the columns readable names for the legend. `px.line`
> with a list `y=[...]` draws one line per column. `log_y=True` gives a ratio scale, `tickvals` sets the
> numbers printed on the axis, and on a ratio scale the axis `range` is given in powers of ten: `np.log10(0.9)`.

In [26]:
us_levels = gdp.rename(columns={"gdp_actual": "Actual GDP", "gdp_potential": "Potential GDP"})
TICKS = [1, 2, 4, 8, 16, 32]
LEVEL_COUNTRIES = ["United States"] + sorted(others.dropna(subset=["gdp_real"]).country.unique())


def gdp_levels(country):
    """Fig. 9.2: actual and potential GDP of one country (U.S.: CBO, 2022 dollars; others: OECD, 2021 PPP dollars)."""
    if country == "United States":
        return us_levels
    data = others[others.country == country].dropna(subset=["gdp_real"])
    return data.rename(columns={"gdp_real": "Actual GDP", "gdp_potential": "Potential GDP"})


def ratio_ticks(low, high):
    """Tick values 1, 2, 5, 10, 20, ... (more of them for a narrow range) from low to high."""
    steps = [1, 2, 5] if high / low > 8 else [1, 1.5, 2, 3, 5, 7]
    if high / low < 3:
        steps = [1, 1.2, 1.5, 2, 2.5, 3, 4, 5, 6, 7, 8]       # a narrow range needs more ticks
    ticks = []
    for power in range(-3, 3):
        for step in steps:
            if low <= step * 10 ** power <= high:
                ticks.append(round(step * 10 ** power, 3))
    return ticks

> **Python note.** `.idxmin()` gives the row label of the smallest value, and `.notna().any()` is True when a
> column has at least one number.

In [27]:
def gdp_title(country, data):
    """Fig. 9.2: a title with the finding: the deepest slump below potential output."""
    gaps = short_run_output(data["Actual GDP"], data["Potential GDP"])
    if country == "United States":
        return "Actual and potential GDP move together, except in the 1930s and 1940s"
    if gaps.notna().any():
        return f"{country}: the deepest slump was {-gaps.min():.1f}% below potential, in {data.year[gaps.idxmin()]}"
    return f"{country}: actual GDP only (the OECD estimates no potential output)"

In [28]:
def gdp_chart(scale="ratio", country="United States"):
    """Fig. 9.2: real GDP, actual and potential, of one country (ratio or linear scale)."""
    data = gdp_levels(country)
    unit = "Trillions of 2022 dollars" if country == "United States" else "Trillions of 2021 dollars (PPP)"
    title = gdp_title(country, data)
    fig = px.line(data, x="year", y=["Actual GDP", "Potential GDP"], log_y=(scale == "ratio"), title=title,
                  color_discrete_sequence=[BLUE, GREEN], labels={"year": "Year", "value": unit, "variable": "Series"})
    fig.update_traces(selector=dict(name="Potential GDP"), line_dash="dash")
    fig.update_traces(selector=dict(name="Actual GDP"), line_width=3)
    if country == "United States":
        ticks, ratio_range, linear_top = TICKS, [np.log10(0.9), np.log10(32)], 30
    else:
        low, high = 0.9 * data["Actual GDP"].min(), 1.1 * data["Actual GDP"].max()
        ticks, ratio_range, linear_top = ratio_ticks(low, high), [np.log10(low), np.log10(high)], 1.05 * high
    if scale == "ratio":
        fig.update_yaxes(tickvals=ticks, range=ratio_range)
    else:
        fig.update_yaxes(range=[0, linear_top])
    fig.update_xaxes(range=[data.year.min() - 1, 2026])
    return fig

In [29]:
interact(gdp_chart, scale=ToggleButtons(options=["ratio", "linear"], value="ratio", description="vertical scale"),
         country=Dropdown(options=LEVEL_COUNTRIES, value="United States", description="country", style=WIDE));

In [30]:
# @title Website version: Fig. 2 with a country menu and buttons for the vertical scale
figures, traces, owner = {}, [], []
for c in LEVEL_COUNTRIES:
    figures[c] = gdp_chart("ratio", c)
    for t in figures[c].data:
        t.visible = (c == "United States")
        traces.append(t)
        owner.append(c)
fig = go.Figure(data=traces, layout=figures["United States"].layout)
menu = []
for c in LEVEL_COUNTRIES:
    lay = figures[c].layout
    shown = []
    for o in owner:
        shown.append(o == c)
    menu.append(dict(label=c, method="update", args=[{"visible": shown}, {
        "title.text": lay.title.text, "yaxis.type": "log", "yaxis.range": list(lay.yaxis.range),
        "yaxis.tickvals": list(lay.yaxis.tickvals), "yaxis.title.text": lay.yaxis.title.text,
        "xaxis.range": list(lay.xaxis.range)}]))
fig.update_layout(updatemenus=[
    dict(buttons=menu, x=0.01, y=0.98, xanchor="left", yanchor="top"),
    dict(type="buttons", direction="right", x=0.99, y=0.02, xanchor="right", yanchor="bottom", buttons=[
        dict(label="ratio scale", method="relayout", args=[{"yaxis.type": "log", "yaxis.autorange": True}]),
        dict(label="linear scale", method="relayout", args=[{"yaxis.type": "linear", "yaxis.autorange": True}])])])
fig

**Sources.** Data: [U.S. Bureau of Economic Analysis (BEA), Real Gross Domestic Product (`GDPCA`), via FRED](https://fred.stlouisfed.org/series/GDPCA);
[Congressional Budget Office (CBO), Real Potential Gross Domestic Product (`GDPPOT`), via FRED](https://fred.stlouisfed.org/series/GDPPOT); and the [BEA's GDP implicit price deflator (`A191RD3A086NBEA`), via FRED](https://fred.stlouisfed.org/series/A191RD3A086NBEA),
used to express both in 2022 dollars. No agency publishes potential output before 1949, so the 1929–1948 values are our own smooth trend.
Other countries: OECD, *Economic Outlook* No. 119, real GDP at purchasing power parity (`GDPVD`) and potential output
(`GDPVTR`, `GDPV`), from the [OECD Data Explorer](https://data-explorer.oecd.org/) (dataflow `DSD_EO@DF_EO`).
Prepared files: [`data/ch09/fig9-2.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/fig9-2.csv) and [`data/ch09/short_run_panel.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/short_run_panel.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/README.md)).
Chart: inspired by Fig. 9.2 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

*On the website, choose a country in the menu at the top left; choosing a country switches back to the ratio scale.*

**What to notice.**

- Since about 1950 the two lines are hard to tell apart. Growth is so large that the fluctuations become
  hard to see, an illusion of scale that Fig. 3 removes.
- On the linear scale, everything before 1960 is squeezed near the bottom.
- Potential output grows more slowly after the Great Recession of 2007–09 (see the key numbers below). The
  CBO builds potential output with a Solow-type model, so weak investment after 2008 meant less capital
  and slower potential growth.
- Choose Greece. After 2008 its GDP fell far below potential, to 10.6 percent below in 2012, and the OECD now estimates
  that Greece's potential output itself shrank, by 0.5 percent a year from 2007 to 2025. Choose Japan: potential output
  has grown by less than 1 percent a year since 1995.

#### Key numbers: the Depression, the war and slower potential growth

> **Python note.** `set_index("year")` labels the rows by year, so `potential[2007]` is the value for 2007.
> `.loc[1930:1939]` keeps the rows from 1930 to 1939, and `.mean()` averages them.

In [31]:
by_year = gdp.set_index("year")
potential = by_year.gdp_potential


def potential_growth(start, end):
    """Average annual growth of potential output from year start to year end, in percent."""
    return 100 * ((potential[end] / potential[start]) ** (1 / (end - start)) - 1)


print(f"Average gap in the 1930s:           {by_year.gap.loc[1930:1939].mean():+.1f}%")
print(f"Gap in 1944 (World War II):         {by_year.gap[1944]:+.1f}%")
print(f"Largest gap since 1950:             {by_year.gap.loc[1950:].min():+.1f}% in {by_year.gap.loc[1950:].idxmin()}")
print(f"Potential growth 1990-2007:         {potential_growth(1990, 2007):.1f}% a year")
print(f"Potential growth 2007-2019:         {potential_growth(2007, 2019):.1f}% a year")
print(f"Real GDP in 2022:                   {by_year.gdp_actual[2022]:.2f} trillion dollars")

Average gap in the 1930s:           -11.4%
Gap in 1944 (World War II):         +35.1%
Largest gap since 1950:             -6.2% in 1982
Potential growth 1990-2007:         3.0% a year
Potential growth 2007-2019:         1.8% a year
Real GDP in 2022:                   26.06 trillion dollars


Output was on average 11 percent below potential in the 1930s. Since 1950 no gap has been as large as
the 1982 one, 6.2 percent below potential. Potential output grew by about 3.0 percent a year from 1990
to 2007 and by only 1.8 percent a year from 2007 to 2019. And real GDP in 2022 was 26 trillion
dollars, so the textbook's "more than 25 trillion dollars" still holds after the data revisions of 2023.

**Your turn.** Find the year after 1950 in which actual output was furthest *above* potential, and how far.

<details><summary>Hint</summary>

Use `by_year.gap.loc[1950:]` as above, with `.max()` and `.idxmax()` instead of `.min()` and `.idxmin()`.

</details>

In [32]:
# Your turn: try it here

### Fig. 3: U.S. economic fluctuations

**Goal.** Make the fluctuations visible by plotting short-run output $\tilde{Y}_t$, quarter by quarter, with the
recessions shaded.

The data file has quarterly real GDP and CBO potential GDP since 1970. The recessions are shaded with
`shade_recessions` from "Model overview".

> **Python note.** `parse_dates=["date"]` turns text such as `"2020-04-01"` into real dates, so that
> Plotly and pandas can work with them. `.tail()` shows the last five rows.

In [33]:
gap = pd.read_csv(DATA_URL + "fig9-3.csv", parse_dates=["date"])
gap.tail()

,date,short_run_output,short_run_output_book_vintage,gdp_real,gdp_potential
221,2025-04-01,1.42703,NaN,23884.6,23548.5
222,2025-07-01,1.82775,NaN,24113.0,23680.2
223,2025-10-01,1.31978,NaN,24125.5,23811.2
224,2026-01-01,1.37500,NaN,24274.4,23945.1
225,2026-04-01,1.40033,NaN,24408.0,24070.9


`short_run_output` is $\tilde{Y}_t$ in percent, computed from the two levels `gdp_real` and `gdp_potential`.
Let us check that with our own function:

In [34]:
check = short_run_output(gap.gdp_real, gap.gdp_potential) - gap.short_run_output
print(f"Largest difference from our own calculation: {check.abs().max():.4f} percentage points")

Largest difference from our own calculation: 0.0008 percentage points


The column `short_run_output_book_vintage` is the *same* calculation with the data **as they were published in
2023**, when the textbook was written. It stops in 2023, which is why the last rows show `NaN` ("not a
number", pandas' mark for a missing value). Since then, the statistical agencies have revised their numbers:
the Bureau of Economic Analysis (BEA) revised real GDP and the CBO re-estimated potential output. Plotting both
vintages shows how much revisions matter.

The OECD's annual short-run output of other countries can go on the same chart. Each of their points is a whole
year, drawn in the middle of the year. The shaded bands stay the U.S. recessions.

> **Python note.** The shaded bands are the recessions. `fig.add_hline` draws a horizontal line, here at
> $\tilde{Y} = 0$. `pd.to_datetime(... + "-07-01")` turns a year into the date of 1 July of that year.

In [35]:
vintages = gap.rename(columns={"short_run_output": "Data as of 2026",
                               "short_run_output_book_vintage": "Data as published in 2023"})
annual_gaps = others.dropna(subset=["output_gap"])[["country", "year", "output_gap"]].copy()
annual_gaps["date"] = pd.to_datetime(annual_gaps.year.astype(str) + "-07-01")   # the middle of each year
GAP_COUNTRIES = ["United States"] + sorted(annual_gaps.country.unique())
GAP_RANGES = [[-11, 5], [-13, 19]]                 # the textbook's range, and a wider one for other countries

In [36]:
def fluctuations_chart(countries=("United States",)):
    """Fig. 9.3: short-run output; U.S. quarterly (two data vintages), other countries annual (OECD)."""
    fig = px.line(vintages, x="date", y=["Data as of 2026", "Data as published in 2023"], color_discrete_sequence=[BLUE, GRAY],
                  labels={"date": "Year", "value": "Short-run output, Ỹ (percent)", "variable": "Data vintage"},
                  title="Short-run output falls sharply in every recession (shaded)")
    fig.update_traces(selector=dict(name="Data as of 2026"), line_width=3)
    fig.update_traces(selector=dict(name="Data as published in 2023"), line_dash="dot")
    if "United States" not in countries and len(countries) > 0:
        fig.update_traces(visible=False)
    data = annual_gaps[annual_gaps.country.isin(countries)]
    if len(data) > 0:
        extra = px.line(data, x="date", y="output_gap", color="country", markers=True, hover_name="year", color_discrete_map=COLORS)
        fig.add_traces(extra.data)
        low = data.loc[data.output_gap.idxmin()]
        fig.update_layout(title=f"U.S. recessions shaded. Deepest slump shown: {low.country}, {low.output_gap:.1f}% in {low.year}")
    fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)
    shade_recessions(fig, "1970-01-01")
    fig.update_xaxes(range=["1970-01-01", "2026-07-01"])
    wide = len(data) > 0 and (data.output_gap.min() < -11 or data.output_gap.max() > 5)
    fig.update_yaxes(range=GAP_RANGES[1] if wide else GAP_RANGES[0], tickvals=None if wide else [-10, -8, -6, -4, -2, 0, 2, 4])
    if len(data) == 0:                             # the textbook's view: the legend fits inside the plot
        fig.update_layout(legend=dict(x=0.99, y=0.99, xanchor="right", yanchor="top", bgcolor="rgba(255,255,255,0.8)"))
    fig.update_layout(legend_title_text="")
    return fig

> **Python note.** `SelectMultiple` is a list from which you can pick several countries: hold Ctrl (Cmd on a Mac)
> and click. The function receives the chosen names as a *tuple*, a list that cannot be changed.

In [37]:
interact(fluctuations_chart, countries=SelectMultiple(options=GAP_COUNTRIES, value=["United States"], rows=8,
                                                      description="countries"));

In [38]:
# @title Website version: Fig. 3 with the other countries in the legend
fig = fluctuations_chart()
extra = px.line(annual_gaps, x="date", y="output_gap", color="country", markers=True, hover_name="year",
                color_discrete_map=COLORS)
add_legend_traces(fig, extra.data)
n_us, n_all = 2, len(fig.data)
fig.update_yaxes(autorange=True)                    # the axis fits the countries you switch on
fig.update_layout(legend=dict(x=1.02, y=1, xanchor="left", yanchor="top"), margin=dict(t=120),
                  updatemenus=[dict(type="buttons", direction="right", x=0, y=1.02, xanchor="left", yanchor="bottom",
                                    buttons=[
    dict(label="the textbook's view", method="restyle", args=[{"visible": [True] * n_us + ["legendonly"] * (n_all - n_us)}]),
    dict(label="all countries", method="restyle", args=[{"visible": [True] * n_all}])])])
fig

**Sources.** Data: [U.S. Bureau of Economic Analysis (BEA), Real Gross Domestic Product (`GDPC1`), via FRED](https://fred.stlouisfed.org/series/GDPC1); [Congressional Budget Office (CBO), Real Potential Gross Domestic Product (`GDPPOT`), via FRED](https://fred.stlouisfed.org/series/GDPPOT);
the 2023 vintage of both series from the [ALFRED real-time archive of the St. Louis Fed](https://alfred.stlouisfed.org/series?seid=GDPC1);
recessions: NBER business-cycle dates, from the [NBER-based recession indicator (`USREC`), via FRED](https://fred.stlouisfed.org/series/USREC) (see the [NBER's dating page](https://www.nber.org/research/business-cycle-dating));
other countries: OECD, *Economic Outlook* No. 119, output gap (`GAP`), from the [OECD Data Explorer](https://data-explorer.oecd.org/) (dataflow `DSD_EO@DF_EO`).
Prepared files: [`data/ch09/fig9-3.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/fig9-3.csv), [`data/ch09/recessions.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/recessions.csv) and [`data/ch09/short_run_panel.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/short_run_panel.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/README.md)).
Chart: inspired by Fig. 9.3 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

*On the website, click a country in the legend to add its annual short-run output, or show them all with the button.*

**What to notice.**

- Every recession begins near a peak of $\tilde{Y}$ and ends close to its trough. The deepest dips are 1982,
  2009 and above all the second quarter of 2020, when Covid-19 closed much of the economy.
- Most of the time $\tilde{Y}$ stays between $-4$ and $+4$ percent (see the key numbers).
- The two vintages tell the same story with different numbers. With today's data the 2020 collapse is
  8.8 percent below potential instead of the 10.4 percent in the textbook, and $\tilde{Y}$ has been positive
  since the second quarter of 2021. Click a name in the legend to hide or show that line.
- Add Latvia and Estonia. Before 2008 they boomed, 18 and 14 percent above potential in 2007, and then crashed to
  12 and 11 percent below in 2009–10, swings much larger than any in the United States. Add Greece: its slump after
  the euro crisis lasted a decade, with output 10.6 percent below potential at the bottom, in 2012.

#### Key numbers: the troughs in two data vintages

The textbook says output fell "nearly 8 percent below potential" in 1982, "more than 5 percent" in 2009,
and "more than 10 percent" in 2020. We look for the lowest $\tilde{Y}$ from the start of each recession to one
year after its end, in both vintages.

> **Python note.** With dates as row labels, `series[peak:end]` keeps the rows between two dates.
> `pd.DateOffset(years=1)` adds one year to a date. Each pass of the loop adds one row to the list `rows`,
> which becomes a DataFrame at the end.

In [39]:
series = gap.set_index("date")
rows = []
for peak, trough in zip(recessions.peak, recessions.trough):
    if peak.year >= 1970:
        window = series[peak:trough + pd.DateOffset(years=1)]
        rows.append([peak.strftime("%Y-%m"), window.short_run_output.min(),
                     window.short_run_output_book_vintage.min()])
pd.DataFrame(rows, columns=["Recession starts", "Lowest Ỹ, data of 2026", "Lowest Ỹ, data of 2023"]).round(2)

,Recession starts,"Lowest Ỹ, data of 2026","Lowest Ỹ, data of 2023"
0,1973-11,-4.72,-4.75
1,1980-01,-3.30,-3.47
2,1981-07,-7.29,-7.88
3,1990-07,-2.24,-3.60
4,2001-03,-2.17,-2.56
5,2007-12,-4.98,-5.48
6,2020-02,-8.83,-10.42


The 2023 vintage reproduces the textbook's numbers: $-7.88$ in 1982, $-5.48$ in 2009 and $-10.42$ in 2020.
Today's data put each trough higher, at $-7.29$, $-4.98$ and $-8.83$. Nothing about 2020 has changed: we just
know more now. Potential output is never observed, only estimated, and every new estimate moves the gap.
**When you quote a number, say which vintage of the data it comes from.**

In [40]:
inside = gap.short_run_output.between(-4, 4)
print(f"Quarters with Y~ between -4% and +4%: {100 * inside.mean():.1f}% of {len(gap)} quarters since 1970")

Quarters with Y~ between -4% and +4%: 92.9% of 226 quarters since 1970


#### Key numbers: two agencies, two estimates

The OECD also estimates the U.S. output gap, with its own model of potential output. Our file has it, so we can
compare it with the CBO's, averaged over the four quarters of each year.

> **Python note.** `pd.DataFrame({"OECD": a, "CBO": b})` lines up two series by their labels, here the years.
> `.dropna()` keeps the years in which both exist.

In [41]:
oecd_us = panel[panel.country == "United States"].set_index("year").output_gap
cbo_us = gap.groupby(gap.date.dt.year).short_run_output.mean()
two = pd.DataFrame({"OECD": oecd_us, "CBO": cbo_us}).dropna()
print(f"{two.index.min()}-{two.index.max()}: correlation {two.OECD.corr(two.CBO):.2f}, "
      f"largest difference {(two.OECD - two.CBO).abs().max():.1f} points in {(two.OECD - two.CBO).abs().idxmax()}")
two.loc[[1992, 2000, 2009, 2020, 2025]].round(2)

1985-2025: correlation 0.80, largest difference 2.0 points in 1986


,OECD,CBO
1992,-0.63,-1.14
2000,1.91,1.75
2009,-3.98,-4.66
2020,-4.22,-3.28
2025,-0.06,1.40


The two agencies agree on the direction of every big swing, but not on the size: the correlation is only about
0.8. Potential output is never observed, so every estimate of $\tilde{Y}$ depends on a model. That is worth
remembering when you compare countries: each of their gaps is the OECD's estimate.

#### Key numbers: what does a recession cost?

The textbook adds up the output lost in a slump: about 6 percent of a year's GDP in a typical recession. We can
measure it. We walk through the quarters, and each time $\tilde{Y}$ turns negative we start adding up the shortfall
until output is back at potential. A quarter is a quarter of a year, so each quarter adds $-\tilde{Y}/4$ percent of
a year's GDP.

> **Python note.** `series.short_run_output.items()` hands over one (date, value) pair at a time. `None` means
> "no value yet": `start is None` is true while we are not inside a slump. `total += x` is short for
> `total = total + x`.

In [42]:
slumps = []
start, total = None, 0.0
for date, value in series.short_run_output.items():
    if value < 0:
        if start is None:                      # a new slump begins
            start, total = date, 0.0
        total += -value / 4                    # lost output, percent of a year's GDP
    elif start is not None:                    # output is back at potential: the slump is over
        years = (date - start).days / 365.25
        slumps.append([start.strftime("%Y-%m"), date.strftime("%Y-%m"), years, total])
        start = None
slumps = pd.DataFrame(slumps, columns=["Slump starts", "Back at potential", "Years", "Lost output (% of GDP)"])
slumps[slumps.Years >= 1].round(1)

,Slump starts,Back at potential,Years,Lost output (% of GDP)
0,1970-04,1972-04,2.0,1.6
1,1974-07,1977-07,3.0,7.8
3,1980-01,1987-10,7.7,17.2
4,1990-10,1994-04,3.5,4.6
6,1995-04,1996-04,1.0,0.4
7,2001-04,2004-07,3.2,4.1
8,2008-07,2018-01,9.5,21.3
9,2020-01,2021-04,1.2,3.4


The milder slumps, around the recessions of 1970, 1990–91, 2001 and 2020, cost between about 1.5 and 5 percent of a
year's GDP, and the 1974–75 slump about 8 percent. Two slumps were far more costly, because output stayed
below potential for many years: the early 1980s and the years after the Great Recession of 2007–09.

In dollars, 6 percent of 2022 GDP is a lot of money. With the textbook's U.S. population of 330 million:

In [43]:
loss = 0.06 * by_year.gdp_actual[2022] * 1e12          # 6 percent of 2022 GDP, in dollars
print(f"6% of 2022 GDP: {loss / 1e12:.2f} trillion dollars")
print(f"Per person:     {loss / 330e6:,.0f} dollars")
print(f"Per family of four: {4 * loss / 330e6:,.0f} dollars")

6% of 2022 GDP: 1.56 trillion dollars
Per person:     4,738 dollars
Per family of four: 18,953 dollars


The textbook rounds these to 1.5 trillion, 4,500 and 18,000 dollars. Our numbers are a little higher because
the revised 2022 GDP is a little higher.

**Your turn.** Compute the average of $\tilde{Y}$ over 2010–2019 in both vintages. Which vintage says the 2010s
were a weaker decade?

<details><summary>Hint</summary>

`series.loc["2010":"2019"]` keeps the quarters of 2010–2019. Then take `.mean()` of each of the two columns.

</details>

In [44]:
# Your turn: try it here

### Fig. 4: Changes in U.S. employment

**Goal.** See recessions in the labor market: the monthly percent change in the number of jobs since 1980.

In [45]:
jobs = pd.read_csv(DATA_URL + "fig9-4.csv", parse_dates=["date"])
april_2020 = jobs.set_index("date").emp_change_pct["2020-04-01"]
print(f"{len(jobs)} months; change in April 2020: {april_2020:.2f}%")
jobs.head()

561 months; change in April 2020: -13.57%


,date,emp_change_pct
0,1980-01-01,0.141168
1,1980-02-01,0.091410
2,1980-03-01,0.122135
3,1980-04-01,-0.159351
4,1980-05-01,-0.472212


`emp_change_pct` is the percent change of nonfarm payroll employment from the month before. In a normal month
it is a small positive number. In April 2020 it was $-13.6$ percent, so we need a second view to see the normal
months.

Payroll jobs are a U.S. statistic, counted from the records of firms each month. Other countries measure
employment with a household survey, the labour force survey, usually every quarter. The OECD collects these
numbers, and a wider file has them, with the percent change from the quarter before:

In [46]:
emp_q = pd.read_csv(DATA_URL + "employment_quarterly.csv", parse_dates=["date"])
emp_q[emp_q.date == "2020-04-01"].sort_values("emp_change_pct").head()

,iso3,country,date,employment,emp_change_pct
1298,CRI,Costa Rica,2020-04-01,1.7453,-20.964
1136,CHL,Chile,2020-04-01,7.1692,-19.689
1235,COL,Colombia,2020-04-01,16.7827,-19.529
4174,MEX,Mexico,2020-04-01,44.6780,-19.505
5552,ZAF,South Africa,2020-04-01,14.1913,-13.298


The second quarter of 2020 shows how differently countries went through the Covid-19 recession. Its numbers are
quarterly, so they are not directly comparable with the monthly U.S. bars, but each country's own chart shows its
recessions the same way.

> **Python note.** `px.bar` draws one bar per row. `fig.add_annotation` writes a label with an arrow
> that points at a data point; `ax` and `ay` set where the label sits, in pixels from that point.
> `f"{date:%B %Y}"` prints a date as "April 2020", and `date.quarter` is its quarter of the year (1 to 4).

In [47]:
def employment_data(country):
    """Fig. 9.4: the employment changes of one country (U.S.: monthly payroll jobs; others: quarterly, OECD)."""
    if country == "United States":
        return jobs, "Monthly"
    return emp_q[emp_q.country == country], "Quarterly"

In [48]:
def employment_chart(view="full range", country="United States"):
    """Fig. 9.4: the percent change in employment from the month (or quarter) before."""
    data, freq = employment_data(country)
    low = data.loc[data.emp_change_pct.idxmin()]
    when = f"{low.date:%B %Y}" if country == "United States" else f"{low.date.year} Q{low.date.quarter}"
    title = "Employment grows in most months; 2008–09 and April 2020 stand out"
    if country != "United States":
        title = f"{country}: employment fell most in {when}, by {-low.emp_change_pct:.1f}% in one quarter"
    fig = px.bar(data, x="date", y="emp_change_pct", title=title,
                 labels={"date": "Year", "emp_change_pct": f"{freq} change in employment (percent)"})
    fig.update_traces(marker_color=BLUE, marker_line_width=0)
    fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)
    if view == "full range":
        fig.add_annotation(x=low.date, y=low.emp_change_pct, text=f"{when}: {low.emp_change_pct:.1f}%",
                           showarrow=True, arrowhead=2, ax=-110, ay=0)
        if country == "United States":
            fig.update_yaxes(range=[-15, 4.5], tickvals=[-14, -12, -10, -8, -6, -4, -2, 0, 2, 4])
    else:
        fig.update_yaxes(range=[-1, 2], tickvals=[-0.5, 0, 0.5, 1, 1.5])   # no label in the bottom-left corner
    fig.update_xaxes(range=[data.date.min() - pd.DateOffset(months=6), data.date.max() + pd.DateOffset(months=6)])
    return fig

In [49]:
interact(employment_chart, view=Dropdown(options=["full range", "zoom on normal months"], value="full range",
                                         description="view", style=WIDE),
         country=Dropdown(options=["United States"] + sorted(emp_q.country.unique()), value="United States",
                          description="country", style=WIDE));

In [50]:
# @title Website version: Fig. 4 with a country menu and buttons for the full range and the zoom
EMP_COUNTRIES = ["United States"] + sorted(emp_q.country.unique())
fig = go.Figure(layout=employment_chart().layout)
menu = []
for c in EMP_COUNTRIES:
    f = employment_chart("full range", c)
    fig.add_trace(f.data[0].update(visible=(c == "United States")))
    shown = []
    for k in range(len(EMP_COUNTRIES)):
        shown.append(EMP_COUNTRIES[k] == c)
    menu.append(dict(label=c, method="update", args=[{"visible": shown}, {
        "title.text": f.layout.title.text, "annotations": [a.to_plotly_json() for a in f.layout.annotations],
        "yaxis.autorange": True, "yaxis.tickvals": None, "xaxis.range": list(f.layout.xaxis.range),
        "yaxis.title.text": f.layout.yaxis.title.text}]))
fig.update_layout(updatemenus=[
    dict(buttons=menu, x=0.45, y=0.98, xanchor="left", yanchor="top"),       # next to the view buttons
    dict(type="buttons", direction="right", x=0.01, y=0.98, xanchor="left", yanchor="top", buttons=[
        dict(label="full range", method="relayout",
             args=[{"yaxis.autorange": True, "yaxis.tickvals": None, "annotations[0].visible": True}]),
        dict(label="zoom on normal months", method="relayout",
             args=[{"yaxis.range": [-1, 2], "yaxis.tickvals": [-0.5, 0, 0.5, 1, 1.5], "annotations[0].visible": False}])])])
fig

**Sources.** Data: [U.S. Bureau of Labor Statistics (BLS), Current Employment Statistics, All Employees, Total Nonfarm (`PAYEMS`), via FRED](https://fred.stlouisfed.org/series/PAYEMS);
other countries: OECD, *Economic Outlook* No. 119, total employment (`ET`, quarterly), from the [OECD Data Explorer](https://data-explorer.oecd.org/)
(dataflow `DSD_EO@DF_EO`).
Prepared files: [`data/ch09/fig9-4.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/fig9-4.csv) and [`data/ch09/employment_quarterly.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/employment_quarterly.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/README.md)).
Chart: inspired by Fig. 9.4 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

*On the website, choose a country in the menu at the top.*

**What to notice.**

- In the full view, April 2020 dwarfs everything else: one month wiped out 13.6 percent of all jobs. The
  rebound came quickly, with gains of 2.0 and 3.5 percent in May and June 2020.
- Zoom in. Now you see small negative bars in the recessions of the early 1980s, 1990–91 and 2001, and a long
  run of negative months in 2008–09.
- Since 2023 the monthly gains have been small: on average 0.13 percent a month in 2023 and close to zero in 2025,
  with a few negative months in 2025–26.
- Choose Germany: in the second quarter of 2020 its employment fell by only 1.2 percent, against 12.7 percent in the
  United States (household survey) and 11.5 percent in Canada. German firms kept their workers on short-time work
  (*Kurzarbeit*), with the government paying part of the wages, so output fell but jobs did not.

#### Key numbers: jobs lost in recessions

The textbook reasons that a typical recession raises the unemployment rate by 1 or 2 percentage points. With a
labor force of about 160 million people, that is a loss of 1.6 to 3.2 million jobs. Let us count the jobs actually
lost, from the peak month to the lowest employment up to two years later. A wider data file has the number of jobs.

In [51]:
payroll = pd.read_csv(DATA_URL + "payroll_employment.csv", parse_dates=["date"]).set_index("date").payroll_millions
print(f"1 to 2 points of a 160 million labor force: {0.01 * 160:.1f} to {0.02 * 160:.1f} million jobs")
rows = []
for peak, trough in zip(recessions.peak, recessions.trough):
    if peak.year >= 1980:
        lowest = payroll[peak:trough + pd.DateOffset(years=2)].min()
        rows.append([peak.strftime("%Y-%m"), payroll[peak], lowest, payroll[peak] - lowest])
pd.DataFrame(rows, columns=["Recession starts", "Jobs at peak (millions)", "Lowest after", "Jobs lost"]).round(2)

1 to 2 points of a 160 million labor force: 1.6 to 3.2 million jobs


,Recession starts,Jobs at peak (millions),Lowest after,Jobs lost
0,1980-01,90.80,89.52,1.28
1,1981-07,91.60,88.77,2.83
2,1990-07,109.82,108.25,1.56
3,2001-03,132.75,130.15,2.60
4,2007-12,138.40,129.71,8.69
5,2020-02,152.29,130.43,21.87


The four recessions from 1980 to 2001 cost between 1.3 and 2.8 million jobs, just as the textbook's rule of thumb
says. The Great Recession cost 8.7 million jobs and the Covid-19 recession 21.9 million.

**Your turn.** How many months since 1980 had an employment *fall* larger than 0.5 percent, and in which years?

<details><summary>Hint</summary>

`jobs[jobs.emp_change_pct < -0.5]` keeps those months. `len(...)` counts them, and `.date.dt.year` gives their years.

</details>

In [52]:
# Your turn: try it here

### Fig. 5: Inflation in the United States

**Goal.** Check a key fact of the short run: inflation usually falls during a recession.

The textbook uses the annual inflation rate of the consumer price index (CPI). We also load the monthly
12-month inflation rate from a wider data file.

> **Python note.** `pd.to_datetime(inflation.year, format="%Y")` turns a year such as 1980 into the date
> 1 January 1980, so that the annual points line up with the shaded recession months.

In [53]:
inflation = pd.read_csv(DATA_URL + "fig9-5.csv")
inflation["date"] = pd.to_datetime(inflation.year, format="%Y")
monthly = pd.read_csv(DATA_URL + "cpi_monthly.csv", parse_dates=["date"])
monthly[monthly.cpi.isna()]

,date,cpi,inflation_12m
801,2025-10-01,NaN,NaN


One month has no CPI: October 2025. The Bureau of Labor Statistics could not collect prices during the federal
government shutdown of that autumn. The 12-month inflation rate is therefore also missing for that month,
and the chart leaves a small gap. The annual average for 2025 in `fig9-5.csv` fills the month by
interpolation, as the data notes explain.

Inflation in other countries comes from the panel (`inflation`, annual CPI inflation from the World Bank). The
shaded bands stay the U.S. recessions: other countries had their own, at other dates. The vertical axis widens in
steps when a country with higher inflation is added.

In [54]:
other_inflation = others.dropna(subset=["inflation"])[["country", "year", "inflation"]].copy()
other_inflation["date"] = pd.to_datetime(other_inflation.year, format="%Y")
INFLATION_RANGES = [[-2.5, 15.5], [-5, 31], [-10, 62], [-20, 124], [-50, 310]]


def inflation_range(low, high):
    """The first range in INFLATION_RANGES that contains low and high (the widest one if none does)."""
    for bottom, top in INFLATION_RANGES:
        if bottom <= low and high <= top:
            return [bottom, top]
    return INFLATION_RANGES[-1]

In [55]:
def inflation_chart(countries=("United States",)):
    """Fig. 9.5: annual CPI inflation (U.S.: also the monthly 12-month rate), with the U.S. recessions shaded."""
    fig = px.line(inflation, x="date", y="inflation_cpi", markers=True,
                  labels={"date": "Year", "inflation_cpi": "Inflation rate (percent)"},
                  title="Inflation usually peaks near the start of a recession and then falls")
    fig.update_traces(line=dict(color=BLUE, width=3), name="United States, annual", showlegend=True)
    recent = monthly[monthly.date >= "1960-01-01"]
    fig.add_trace(go.Scatter(x=recent.date, y=recent.inflation_12m, name="United States, 12-month rate, monthly",
                             mode="lines", line=dict(color=TEAL, width=1)))
    if "United States" not in countries and len(countries) > 0:
        fig.update_traces(visible=False)
    data = other_inflation[other_inflation.country.isin(countries)]
    if len(data) > 0:
        extra = px.line(data, x="date", y="inflation", color="country", markers=True, hover_name="year", color_discrete_map=COLORS)
        fig.add_traces(extra.data)
        top = data.loc[data.inflation.idxmax()]
        fig.update_layout(title=f"U.S. recessions shaded. Highest inflation shown: {top.country}, {top.inflation:,.1f}% in {top.year}")
    fig.add_hline(y=0, line_color="black", line_width=1)
    shade_recessions(fig)
    fig.update_yaxes(range=inflation_range(min(data.inflation.min(), 0), max(data.inflation.max(), 0)) if len(data) > 0 else [-2.5, 15.5])
    fig.update_xaxes(range=["1959-07-01", "2026-07-01"])
    if len(data) == 0:                             # the textbook's view: the legend fits inside the plot
        fig.update_layout(legend=dict(x=0.99, y=0.99, xanchor="right", yanchor="top", bgcolor="rgba(255,255,255,0.8)"))
    return fig

In [56]:
interact(inflation_chart, countries=SelectMultiple(options=["United States"] + sorted(other_inflation.country.unique()),
                                                   value=["United States"], rows=8, description="countries"));

In [57]:
# @title Website version: Fig. 5 with a menu of country groups and the countries in the legend
fig = inflation_chart()
n_us = len(fig.data)
extra = px.line(other_inflation, x="date", y="inflation", color="country", markers=True, hover_name="year",
                color_discrete_map=COLORS)
add_legend_traces(fig, extra.data)
buttons = []
for group, members in GROUPS_9.items():
    shown = [True] * n_us
    for t in extra.data:
        shown.append("legendonly" if t.name in members else False)
    buttons.append(dict(label=group, method="restyle", args=[{"visible": shown}]))
for t in fig.data[n_us:]:
    t.visible = "legendonly" if t.name in GROUPS_9["G7"] else False
fig.update_yaxes(autorange=True)                    # the axis fits the countries you switch on
fig.update_layout(updatemenus=[dict(buttons=buttons, x=0.01, y=0.98, xanchor="left", yanchor="top")],
                  legend=dict(x=1.02, y=1, xanchor="left", yanchor="top"))
fig

**Sources.** Data: [U.S. Bureau of Labor Statistics (BLS), Consumer Price Index for All Urban Consumers (`CPIAUCSL`), via FRED](https://fred.stlouisfed.org/series/CPIAUCSL), annual and 12-month inflation computed by us;
recessions: NBER business-cycle dates, from the [NBER-based recession indicator (`USREC`), via FRED](https://fred.stlouisfed.org/series/USREC) (see the [NBER's dating page](https://www.nber.org/research/business-cycle-dating));
other countries: [World Bank, World Development Indicators, Inflation, consumer prices (`FP.CPI.TOTL.ZG`)](https://data.worldbank.org/indicator/FP.CPI.TOTL.ZG), with gaps filled from the [World Bank's Global Inflation Database](https://www.worldbank.org/en/research/brief/inflation-database), and OECD *Economic Outlook* No. 119 CPI inflation from the [OECD Data Explorer](https://data-explorer.oecd.org/) where both are missing.
Prepared files: [`data/ch09/fig9-5.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/fig9-5.csv), [`data/ch09/cpi_monthly.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/cpi_monthly.csv), [`data/ch09/recessions.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/recessions.csv) and [`data/ch09/short_run_panel.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/short_run_panel.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/README.md)).
Chart: inspired by Fig. 9.5 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

*On the website, a menu picks a group of countries, and a click on a name in the legend adds that country.*

**What to notice.**

- The peaks of inflation in 1970, 1974, 1980 and 1990 sit at or just before a shaded recession, and inflation
  falls during and after it.
- The highest annual rate is 13.5 percent in 1980, and the lowest $-0.3$ percent in 2009, in the Great Recession.
- The 2020 recession lasted only two months, so the annual points barely notice it. The monthly line does
  dip in spring 2020. Then came the surge of 2021–22.
- Add Japan, Germany and the United Kingdom. The oil shocks raised inflation everywhere, but by very different
  amounts: Japan peaked at 23.2 percent in 1974 and the United Kingdom at 24.2 percent in 1975, while Germany peaked
  at 7.0 percent in 1973. The shading shows U.S. recessions only, so it does not date the other countries' slumps.

#### Key numbers: inflation before and after each recession

In [58]:
pi = inflation.set_index("year").inflation_cpi
rows = []
for peak, trough in zip(recessions.peak, recessions.trough):
    rows.append([peak.year, pi[peak.year], pi.loc[peak.year:trough.year].max(), pi[trough.year + 1]])
pd.DataFrame(rows, columns=["Recession starts", "Inflation that year", "Highest in the recession",
                            "Year after it ends"]).round(2)

,Recession starts,Inflation that year,Highest in the recession,Year after it ends
0,1960,1.49,1.49,1.18
1,1969,5.44,5.88,4.23
2,1973,6.26,11.01,5.78
3,1980,13.50,13.50,10.38
4,1981,10.38,10.38,3.16
5,1990,5.42,5.42,3.04
6,2001,2.82,2.82,1.60
7,2007,2.87,3.82,1.64
8,2020,1.25,1.25,4.68


In every recession before 2020, inflation in the year after the recession ended was lower than at its start.
The Covid-19 recession is the exception in annual data: 1.25 percent in 2020, 4.68 percent in 2021. The monthly
data tell a different story:

In [59]:
rate = monthly.set_index("date").inflation_12m
print(f"12-month inflation, February 2020: {rate['2020-02-01']:.2f}%   April 2020: {rate['2020-04-01']:.2f}%")
print(f"Inflation 1979: {pi[1979]:.2f}%   1980: {pi[1980]:.2f}%   1983: {pi[1983]:.2f}%")

12-month inflation, February 2020: 2.34%   April 2020: 0.31%
Inflation 1979: 11.25%   1980: 13.50%   1983: 3.16%


From February to April 2020, the two months of the Covid-19 recession, 12-month inflation fell from 2.34 to
0.31 percent, so the rule holds even there. The second line is the Volcker disinflation that the textbook uses
as its main example (Fig. 6): inflation of 11 to 13 percent in 1979–80 fell to about 3 percent by 1983.

**Your turn.** Find the highest monthly 12-month inflation rate since 2000, and the month it happened.

<details><summary>Hint</summary>

Use `rate.loc["2000":]` and then `.max()` and `.idxmax()`.

</details>

In [60]:
# Your turn: try it here

## The Phillips curve

When output is above potential, inflation tends to rise. We draw this relationship in the model and then measure it in U.S. data.

### Fig. 6: The Phillips curve

**Goal.** Draw the Phillips curve and see how short-run output moves inflation.

Think of a hot holiday toy. When demand is high, the firm that makes it runs overtime and pays its workers
more, and the toy is scarce. So its price rises faster than usual. In a booming economy, it is as if *every* firm
made the hot toy: all of them raise prices faster than the current inflation rate, so inflation goes up. In a
slump the opposite happens. The Phillips curve, equation (5), puts this in one line:

$$\Delta\pi_t = \bar{\nu}\,\tilde{Y}_t$$

When output is at potential ($\tilde{Y} = 0$), inflation is steady. Our function `phillips_curve` from "The model
in code" draws the line. The black dot is the economy, at the short-run output you choose.

> **Python note.** `fig.add_shape(type="line", ...)` draws a straight segment, here the dashed guides from the
> dot to the axes. An `if ... elif ... else` block picks one of three cases.

In [61]:
def phillips_diagram(nu=0.33, Y_tilde=0.0):
    """Fig. 9.6: the Phillips curve delta pi = nu * Y_tilde, with the economy at Y_tilde."""
    Y_end = min(6.0, 2.6 / nu)                          # stop the line where its name still fits below the top
    Y = np.linspace(-Y_end, Y_end, 100)
    dpi = phillips_curve(Y_tilde, nu)
    if dpi > 0:
        verdict = "inflation rises"
    elif dpi < 0:
        verdict = "inflation falls"
    else:
        verdict = "inflation is steady"
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=Y, y=phillips_curve(Y, nu), name="Phillips curve", line=dict(color=TEAL, width=4)))
    add_end_label(fig, Y_end, phillips_curve(Y_end, nu), "Phillips curve", TEAL, "bottom", xanchor="right")
    fig.add_trace(go.Scatter(x=[Y_tilde], y=[dpi], name="The economy", mode="markers",
                             marker=dict(color="black", size=13)))
    fig.add_shape(type="line", x0=Y_tilde, y0=-3.2, x1=Y_tilde, y1=dpi, line=dict(color=GRAY, dash="dash"))
    fig.add_shape(type="line", x0=-6.5, y0=dpi, x1=Y_tilde, y1=dpi, line=dict(color=GRAY, dash="dash"))
    fig.add_annotation(x=-3.2, y=2.8, text="← Ỹ < 0: output below potential", showarrow=False)
    fig.add_annotation(x=3.2, y=-2.8, text="Ỹ > 0: output above potential →", showarrow=False)
    fig.update_layout(title=f"Ỹ = {Y_tilde:+.2f}%  →  Δπ = {nu:.2f} × ({Y_tilde:+.2f}) = {dpi:+.2f} points: {verdict}",
                      xaxis=dict(title="Short-run output, Ỹ (percent)", range=[-6.5, 6.5], tickvals=[-4, -2, 0, 2, 4, 6]),
                      yaxis=dict(title="Change in inflation, Δπ (points)", range=[-3.2, 3.2], tickvals=[-2, -1, 0, 1, 2, 3]),
                      showlegend=False)                  # tick values: no labels in the bottom-left corner
    return fig

In [62]:
interact(phillips_diagram,
         nu=FloatSlider(value=0.33, min=0.05, max=0.5, step=0.01, description="slope ν", style=WIDE),
         Y_tilde=FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.5, description="short-run output Ỹ (%)", style=WIDE));

In [63]:
# @title Website version: Fig. 6 with a precomputed slider for short-run output
slider_figure(phillips_diagram, "Y_tilde", [-4.0, -3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 4.0], start=0.0,
              label="short-run output Ỹ (percent)")

**Source.** Adapted from Fig. 9.6 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- To the right of zero (a boom), $\Delta\pi > 0$: inflation rises. To the left (a slump), $\Delta\pi < 0$: inflation falls.
- The line always goes through the origin: only output at potential keeps inflation steady.
- A steeper curve (a larger $\bar{\nu}$) means a boom raises inflation faster, but also that a slump brings it
  down faster.

#### How the short-run model works: the Volcker disinflation

In 1979 U.S. inflation was above 11 percent (Fig. 5), and the Federal Reserve under Paul Volcker set out to
bring it down. The Phillips curve says that the only way to lower inflation is to push output below potential.
Tight monetary policy did exactly that: short-run output averaged about $-6$ percent in 1982 (Fig. 3 shows the
quarters). The
model then predicts falling inflation, and inflation did fall to about 3 percent by 1983.

Let us check the model's prediction for 1981–1983 with the annual short-run output of Fig. 7 below.

In [64]:
phillips = pd.read_csv(DATA_URL + "fig9-7.csv")
volcker = phillips[(phillips.year >= 1981) & (phillips.year <= 1983)].copy()
volcker["predicted_change"] = phillips_curve(volcker.short_run_output)
volcker[["year", "short_run_output", "d_inflation", "predicted_change"]].round(2)

,year,short_run_output,d_inflation,predicted_change
20,1981,-1.82,-3.12,-0.60
21,1982,-6.22,-4.22,-2.05
22,1983,-4.80,-3.00,-1.58


> **Python note.** `.copy()` makes a separate small table, so that adding a column to `volcker` does not touch
> `phillips`.

The model gets the direction right in all three years. It predicts a smaller fall than actually happened, a
sign that the Phillips curve is not the whole story (Fig. 7).

**Your turn.** With the baseline slope $\bar{\nu} = 0.33$, how far below potential must output be to lower
inflation by 2 percentage points in one year? Find it with the sliders, then check it with `phillips_curve`.

<details><summary>Hint</summary>

Solve $-2 = 0.33 \times \tilde{Y}$ for $\tilde{Y}$. The slider for $\tilde{Y}$ stops at $-5$, so the exact answer
needs the code cell.

</details>

In [65]:
# Your turn: try it here

### Fig. 7: Measuring the Phillips curve

**Goal.** Test the Phillips curve with U.S. data: each point is one year, with that year's short-run output and
change in inflation.

We loaded `fig9-7.csv` above as `phillips`. Its columns are `short_run_output` ($\tilde{Y}$, the average of the four
quarters), `inflation_cpi` ($\pi$), `d_inflation` ($\Delta\pi_t = \pi_t - \pi_{t-1}$) and `d_inflation_fitted`.

In [66]:
phillips.head()

,year,short_run_output,inflation_cpi,d_inflation,d_inflation_fitted
0,1961,-2.300,1.070,-0.422,-0.710
1,1962,-0.233,1.176,0.106,0.020
2,1963,-0.191,1.256,0.080,0.035
3,1964,1.125,1.322,0.066,0.500
4,1965,3.144,1.579,0.257,1.213


> **Python note.** `np.polyfit(x, y, 1)` fits the best straight line $y = a + bx$ through the points (by least
> squares, also called OLS) and returns the slope $b$ first and the intercept $a$ second.

In [67]:
slope, intercept = np.polyfit(phillips.short_run_output, phillips.d_inflation, 1)
print(f"OLS line: delta pi = {intercept:.2f} + {slope:.2f} x Y~   ({phillips.year.min()}-{phillips.year.max()})")

OLS line: delta pi = 0.10 + 0.35 x Y~   (1961-2025)


The fitted slope is about 0.35, close to the textbook's "about 1/3". Now we draw the data with our *model* line
$\Delta\pi = \bar{\nu}\tilde{Y}$, and you choose $\bar{\nu}$. The title adds up the **squared misses**, the squared vertical
distances between the points and the line. The smaller that number, the better the line fits.

> **Python note.** `.where(condition, "")` keeps a value where the condition is true and puts `""` elsewhere, so
> only a few years get a text label (hover over any point to see its year). `np.where(condition, a, b)` picks `a`
> or `b` row by row.

In [68]:
LABELS = [1966, 1974, 1975, 1979, 1982, 1983, 2009, 2020, 2021, 2022, 2023, 2024, 2025]
phillips["label"] = phillips.year.astype(str).where(phillips.year.isin(LABELS), "")
phillips["period"] = np.where(phillips.year <= 2022, "1961–2022 (in the textbook)", "2023–2025 (new data)")
X_RANGES, Y_RANGES = [[-8.5, 6.5], [-13, 19]], [[-5.5, 5.5], [-12, 12]]   # the textbook's ranges, and wider ones


def wider(values, ranges):
    """The first range that holds all values (the last one if none does)."""
    for low, high in ranges:
        if low <= values.min() and values.max() <= high:
            return [low, high]
    return ranges[-1]

For another country, the points are the OECD's output gap and the change in World Bank CPI inflation.

> **Python note.** `.diff()` gives the change from the row before: here, $\Delta\pi_t = \pi_t - \pi_{t-1}$.

In [69]:
def phillips_data(country):
    """Fig. 9.7: one point per year, short-run output and the change in inflation (U.S.: the textbook's data)."""
    if country == "United States":
        return phillips
    data = others[others.country == country].copy()
    data["d_inflation"] = data.inflation.diff()
    data = data.rename(columns={"output_gap": "short_run_output"}).dropna(subset=["short_run_output", "d_inflation"])
    data["label"] = ""
    data["period"] = f"{data.year.min()}–{data.year.max()} (OECD data)"
    return data

In [70]:
def phillips_fit(nu=0.33, country="United States"):
    """Fig. 9.7: the data with the model line delta pi = nu * Y_tilde; the title measures the fit."""
    data = phillips_data(country)
    misses = data.d_inflation - phillips_curve(data.short_run_output, nu)
    ols = np.polyfit(data.short_run_output, data.d_inflation, 1)[0]
    fig = px.scatter(data, x="short_run_output", y="d_inflation", color="period", text="label", hover_name="year",
                     color_discrete_sequence=[GREEN, "#0b4f47"],
                     labels={"short_run_output": "Short-run output, Ỹ (percent)",
                             "d_inflation": "Change in inflation, Δπ (points)", "period": "Years"})
    fig.update_traces(textposition="middle right", textfont_size=11, marker_size=9)
    xr, yr = wider(data.short_run_output, X_RANGES), wider(data.d_inflation, Y_RANGES)
    Y = np.linspace(xr[0], xr[1], 50)                # the model line spans the whole x-axis
    fig.add_trace(go.Scatter(x=Y, y=phillips_curve(Y, nu), name=f"Model: Δπ = {nu:.2f} × Ỹ", mode="lines",
                             line=dict(color=BLUE, width=3)))
    off = (~data.d_inflation.between(yr[0], yr[1]) | ~data.short_run_output.between(xr[0], xr[1])).sum()
    note = f"  |  {off} off the chart" if off > 0 else ""
    fig.update_layout(title=f"{country}, ν = {nu:.2f}: squared misses {(misses ** 2).sum():,.1f}  |  OLS slope {ols:.2f}{note}",
                      xaxis_range=xr, yaxis_range=yr)
    return fig

The axes widen once when a country's points need more room; points beyond even the wider axes are counted in
the title.

In [71]:
interact(phillips_fit, nu=FloatSlider(value=0.33, min=0.0, max=1.0, step=0.01, description="model slope ν", style=WIDE),
         country=Dropdown(options=GAP_COUNTRIES, value="United States", description="country", style=WIDE));

In [72]:
# @title Website version: Fig. 7 with a precomputed slider for the model slope and the other countries in the legend
fig = slider_figure(phillips_fit, "nu", [0.0, 0.1, 0.2, 0.25, 0.33, 0.4, 0.5, 0.75, 1.0], start=0.33, label="model slope ν")
points = []
for c in GAP_COUNTRIES[1:]:
    data = phillips_data(c)
    data["country"] = c
    points.append(data)
extra = px.scatter(pd.concat(points), x="short_run_output", y="d_inflation", color="country", hover_name="year",
                   color_discrete_map=COLORS)
add_legend_traces(fig, extra.data)
fig

**Sources.** Data: [U.S. Bureau of Economic Analysis (BEA), Real Gross Domestic Product (`GDPC1`), via FRED](https://fred.stlouisfed.org/series/GDPC1); [Congressional Budget Office (CBO), Real Potential Gross Domestic Product (`GDPPOT`), via FRED](https://fred.stlouisfed.org/series/GDPPOT); [U.S. Bureau of Labor Statistics (BLS), Consumer Price Index for All Urban Consumers (`CPIAUCSL`), via FRED](https://fred.stlouisfed.org/series/CPIAUCSL);
other countries: OECD, *Economic Outlook* No. 119, output gap (`GAP`), from the [OECD Data Explorer](https://data-explorer.oecd.org/) (dataflow `DSD_EO@DF_EO`), and [World Bank, World Development Indicators, Inflation, consumer prices (`FP.CPI.TOTL.ZG`)](https://data.worldbank.org/indicator/FP.CPI.TOTL.ZG), with gaps filled from the [World Bank's Global Inflation Database](https://www.worldbank.org/en/research/brief/inflation-database).
Prepared files: [`data/ch09/fig9-7.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/fig9-7.csv) and [`data/ch09/short_run_panel.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/short_run_panel.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/README.md)).
Chart: inspired by Fig. 9.7 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

*On the website, click a country in the legend to add its points. The title's numbers are for the United States.
Points beyond the axes appear when you click "Autoscale" in the toolbar at the top right of the chart.*

**What to notice.**

- The cloud slopes upward: years with output above potential tend to have rising inflation.
- Move the slider. The squared misses are smallest for a slope of about 0.35, close to the OLS slope.
- Many points lie far from any line. The oil shocks of 1974 and 1979 and the inflation surge of 2021–22 raised
  inflation by much more than output alone predicts. In 2023–2025 inflation *fell* while output was above potential.
  Shocks move inflation too, and later chapters add them to the model.
- Choose other countries. Most OECD slopes are positive, between about 0.1 and 0.6 (Germany 0.42, Japan 0.20, France
  0.16). Mexico's is negative (−0.90): in 1995 a currency crisis pushed output 6 percent below potential and *raised*
  inflation by 28 points. A price shock like that moves the points off any Phillips curve.

#### Key numbers: the textbook's points in today's data

The textbook says that in 2020 output was about 5 percent below potential and inflation fell by about 1 point.
In today's data the 2020 point sits much further right. The quarterly file of Fig. 3 has both vintages, so we
can compute the annual average of $\tilde{Y}$ for each.

> **Python note.** `groupby(gap.date.dt.year)` puts the quarters of each year in one group, and `.mean()` averages
> each group. `merge` joins two tables on a common column, here `year`.

In [73]:
annual = gap.groupby(gap.date.dt.year)[["short_run_output", "short_run_output_book_vintage"]].mean()
annual = annual.rename_axis("year").reset_index()
compare = phillips[["year", "d_inflation"]].merge(annual, on="year")
compare[compare.year.isin([2019, 2020, 2021, 2022])].round(2)

,year,d_inflation,short_run_output,short_run_output_book_vintage
49,2019,-0.63,0.74,-0.28
50,2020,-0.56,-3.28,-4.78
51,2021,3.43,0.81,-0.86
52,2022,3.31,0.92,-0.67


In the 2023 data, 2020 sits at $\tilde{Y} = -4.78$ with $\Delta\pi = -0.56$, as in the textbook's figure. With
today's data the same year sits at $-3.28$, and 2021–22 have moved from slightly below potential to about
1 percent above it. Data revisions can move a point across the vertical axis.

**Your turn.** Fit the OLS line again with the textbook's years only (1961–2022). Does the slope change much?

<details><summary>Hint</summary>

Make `old = phillips[phillips.year <= 2022]` and call `np.polyfit(old.short_run_output, old.d_inflation, 1)`.

</details>

In [74]:
# Your turn: try it here

## Okun's law

Output below potential also means more people out of work. Okun's law puts a number on that link.

### Fig. 8: Okun's law for the U.S. economy

**Goal.** Link short-run output to unemployment with Okun's law (equation 8), $u - \bar{u} = -\tfrac{1}{2}\tilde{Y}$.

**Cyclical unemployment** $u - \bar{u}$ is the unemployment rate minus the natural rate, the rate that prevails
in the long run (Chapter 7). Here $\bar{u}$ is the CBO's estimate.

In [75]:
okun = pd.read_csv(DATA_URL + "fig9-8.csv")
gap_check = (okun.unrate - okun.nrou_short_term - okun.cyclical_unemployment).abs().max()
print(f"cyclical_unemployment = unrate - nrou_short_term (largest difference {gap_check:.3f})")
okun.head()

cyclical_unemployment = unrate - nrou_short_term (largest difference 0.001)


,year,short_run_output,unrate,nrou_short_term,cyclical_unemployment,cyclical_unemployment_fitted
0,1960,-0.984,5.542,5.467,0.075,0.850
1,1961,-2.300,6.692,5.495,1.197,1.660
2,1962,-0.233,5.567,5.528,0.039,0.388
3,1963,-0.191,5.642,5.567,0.074,0.362
4,1964,1.125,5.158,5.614,-0.456,-0.448


In [76]:
slope_okun, intercept_okun = np.polyfit(okun.short_run_output, okun.cyclical_unemployment, 1)
print(f"OLS line: u - u_bar = {intercept_okun:.2f} + ({slope_okun:.2f}) x Y~   ({okun.year.min()}-{okun.year.max()})")

OLS line: u - u_bar = 0.24 + (-0.62) x Y~   (1960-2025)


As with the Phillips curve, choose the slope of the model line `okun_law` and watch the squared misses.

For other countries we need a natural rate of unemployment too. The OECD no longer publishes its estimates in
the *Economic Outlook*, so our file has a stand-in: `unemployment_trend`, a smooth trend through each country's
unemployment rate (a Hodrick–Prescott trend). Cyclical unemployment is then $u - \text{trend}$. How much does this
choice matter? We can try it on the United States, where we have both:

In [77]:
us_oecd = panel[panel.country == "United States"].set_index("year")
trial = pd.DataFrame({"Y_tilde": okun.set_index("year").short_run_output,
                      "cyclical": us_oecd.unemployment - us_oecd.unemployment_trend}).dropna()
print(f"Okun slope with the CBO natural rate: {slope_okun:.2f};  with a smooth trend instead: "
      f"{np.polyfit(trial.Y_tilde, trial.cyclical, 1)[0]:.2f}")

Okun slope with the CBO natural rate: -0.62;  with a smooth trend instead: -0.38


The slope is smaller with the trend, because a trend follows unemployment up in long slumps and counts part of the
slump as "natural". Keep this in mind when you compare countries: their slopes use the trend, the U.S. one the CBO.

In [78]:
okun["label"] = okun.year.astype(str).where(okun.year.isin([1966, 1969, 1982, 2009, 2010, 2020, 2021]), "")
OKUN_Y_RANGES = [[-3.5, 4.5], [-7, 9]]          # the textbook's range, and a wider one


def okun_data(country):
    """Fig. 9.8: one point per year, short-run output and cyclical unemployment (U.S.: the textbook's data)."""
    if country == "United States":
        return okun
    data = others[others.country == country].copy()
    data["cyclical_unemployment"] = data.unemployment - data.unemployment_trend
    data = data.rename(columns={"output_gap": "short_run_output"}).dropna(subset=["short_run_output", "cyclical_unemployment"])
    data["label"] = ""
    return data

In [79]:
def okun_fit(slope=-0.5, country="United States"):
    """Fig. 9.8: the data with the model line u - u_bar = slope * Y_tilde; the title measures the fit."""
    data = okun_data(country)
    misses = data.cyclical_unemployment - okun_law(data.short_run_output, slope)
    ols = np.polyfit(data.short_run_output, data.cyclical_unemployment, 1)[0]
    fig = px.scatter(data, x="short_run_output", y="cyclical_unemployment", text="label", hover_name="year",
                     color_discrete_sequence=[GREEN],
                     labels={"short_run_output": "Short-run output, Ỹ (percent)",
                             "cyclical_unemployment": "Cyclical unemployment, u − ū (points)"})
    fig.update_traces(textposition="middle right", textfont_size=11, marker_size=9, showlegend=True,
                      name=f"{country}, {data.year.min()}–{data.year.max()}")
    xr, yr = wider(data.short_run_output, X_RANGES), wider(data.cyclical_unemployment, OKUN_Y_RANGES)
    Y = np.linspace(xr[0], xr[1], 50)                # the model line spans the whole x-axis
    fig.add_trace(go.Scatter(x=Y, y=okun_law(Y, slope), name=f"Model: u − ū = {slope:.2f} × Ỹ", mode="lines",
                             line=dict(color=BLUE, width=3)))
    fig.update_layout(title=f"{country}, slope {slope:.2f}: squared misses {(misses ** 2).sum():.2f}  |  OLS slope {ols:.2f}",
                      xaxis_range=xr, yaxis_range=yr)
    return fig

In [80]:
interact(okun_fit, slope=FloatSlider(value=-0.5, min=-1.0, max=0.0, step=0.01, description="model slope", style=WIDE),
         country=Dropdown(options=GAP_COUNTRIES, value="United States", description="country", style=WIDE));

In [81]:
# @title Website version: Fig. 8 with a precomputed slider for the model slope and the other countries in the legend
fig = slider_figure(okun_fit, "slope", [-1.0, -0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2, 0.0], start=-0.5, label="model slope")
points = []
for c in GAP_COUNTRIES[1:]:
    data = okun_data(c)
    data["country"] = c
    points.append(data)
extra = px.scatter(pd.concat(points), x="short_run_output", y="cyclical_unemployment", color="country",
                   hover_name="year", color_discrete_map=COLORS)
add_legend_traces(fig, extra.data)
fig

**Sources.** Data: [U.S. Bureau of Economic Analysis (BEA), Real Gross Domestic Product (`GDPC1`), via FRED](https://fred.stlouisfed.org/series/GDPC1); [Congressional Budget Office (CBO), Real Potential Gross Domestic Product (`GDPPOT`), via FRED](https://fred.stlouisfed.org/series/GDPPOT);
[BLS, Unemployment Rate (`UNRATE`), via FRED](https://fred.stlouisfed.org/series/UNRATE); [CBO, Noncyclical Rate of Unemployment, short-term (`NROUST`), via FRED](https://fred.stlouisfed.org/series/NROUST);
other countries: OECD, *Economic Outlook* No. 119, output gap (`GAP`), from the [OECD Data Explorer](https://data-explorer.oecd.org/) (dataflow `DSD_EO@DF_EO`), and unemployment rate (`UNR`); the trend through each country's unemployment rate is our own (Hodrick–Prescott).
Prepared files: [`data/ch09/fig9-8.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/fig9-8.csv) and [`data/ch09/short_run_panel.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/short_run_panel.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch09/README.md)).
Chart: inspired by Fig. 9.8 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

*On the website, click a country in the legend to add its points (cyclical unemployment measured from a trend, see
above). The title's numbers are for the United States.*

**What to notice.**

- This cloud is much tighter than the Phillips curve's: output and unemployment move together closely.
- The textbook's rule of thumb is a slope of $-1/2$. The best line through the origin is a bit steeper: try
  $-0.6$ and compare the squared misses. The OLS slope is about $-0.62$.
- 2020 lies far above the line: unemployment jumped much more than the fall in output alone predicts, as
  businesses closed and laid off workers during the pandemic.
- Choose Spain: its slope is −0.59, and cyclical unemployment swings by up to 6 points either way. Japan's slope is
  only −0.13: Japanese firms tend to keep their workers through a slump. Remember that these countries' $\bar{u}$ is a
  trend, which gives smaller slopes than the CBO's natural rate (−0.38 instead of −0.62 for the United States).

#### Key numbers: Okun's law in 2009 and 2020

In [82]:
for year in [2009, 2020]:
    row = okun[okun.year == year].iloc[0]
    print(f"{year}: Y~ = {row.short_run_output:+.2f}%, Okun predicts u - u_bar = "
          f"{okun_law(row.short_run_output):+.2f}, actual {row.cyclical_unemployment:+.2f}")

2009: Y~ = -4.66%, Okun predicts u - u_bar = +2.33, actual +3.94
2020: Y~ = -3.28%, Okun predicts u - u_bar = +1.64, actual +3.60


> **Python note.** `okun[okun.year == year].iloc[0]` picks the one row of that year, and `row.short_run_output`
> reads one value from it.

In both years the rule of thumb falls short. In 2009 it predicts unemployment 2.3 points above the natural rate,
against 3.9 in the data, and in 2020 it predicts 1.6 points against 3.6. Part of the reason is the data revisions of
Fig. 7, which moved both years to the right. The textbook's example, output about 6 percent below potential
with unemployment just over 3 points above normal, fits the rule exactly: $-\tfrac{1}{2} \times (-6) = 3$.

**Your turn.** Okun's law also works backward. In which year since 1960 was cyclical unemployment lowest, and what
short-run output does Okun's law imply for it? Compare with the data.

<details><summary>Hint</summary>

Find the row with `okun.cyclical_unemployment.idxmin()`. Solving $u - \bar{u} = -\tfrac{1}{2}\tilde{Y}$ gives
$\tilde{Y} = -2\,(u - \bar{u})$.

</details>

In [83]:
# Your turn: try it here

## Overstimulating the economy

We end with an experiment: what happens to inflation if policymakers keep output above potential year after year?

### Fig. 9: The Phillips curve and the booming economy

**Goal.** Answer the textbook's exercise "Overstimulating the economy": what happens if policymakers keep output
3 percent above potential every year?

The economy starts at potential with inflation at its long-run rate, $\bar{\pi} = 2\%$. Then output is held at
$\tilde{Y} = +3\%$. The Phillips curve (equation 5) says inflation rises by $\Delta\pi = \bar{\nu}\tilde{Y}$ **every** year:

$$\pi_t = \pi_{t-1} + \bar{\nu}\,\tilde{Y}_t$$

With the textbook's slope $\bar{\nu} = 1$, inflation goes from 2 to 5, 8 and 11 percent in three years, the straight-line climb that equation (6)
predicts for a gap held constant. A computer can apply this equation year after year: a **simulation**.

> **Python note.** A `for` loop repeats the indented lines once for each year. `pi_path[-1]` is the last
> number in the list, last year's inflation, and `pi_path.append(...)` adds this year's at the end.
> `range(1, years + 1)` counts 1, 2, ..., years.

In [84]:
def simulate_inflation(pi_bar=2.0, nu=1.0, Y_tilde=3.0, years=5):
    """Inflation year by year when output is held at Y_tilde: pi_t = pi_(t-1) + nu * Y_tilde."""
    pi_path = [pi_bar]                                    # year 0: inflation at its long-run rate
    for year in range(1, years + 1):
        pi_path.append(pi_path[-1] + phillips_curve(Y_tilde, nu))
    return np.arange(years + 1), np.array(pi_path)


years, pi_path = simulate_inflation()
print("Inflation, years 0 to 5:", pi_path)

Inflation, years 0 to 5: [ 2.  5.  8. 11. 14. 17.]


The figure has two panels: (a) the Phillips curve with the economy at potential and in the boom (the two black
dots), and (b) the inflation path $\pi$ that the boom produces, against long-run inflation $\bar{\pi}$ (dashed).

In [85]:
def booming_economy(pi_bar=2.0, nu=1.0, Y_tilde=3.0):
    """Fig. 9.9: the economy held at Y_tilde on the Phillips curve (a) and the inflation path (b)."""
    years, pi_path = simulate_inflation(pi_bar, nu, Y_tilde)
    dpi = phillips_curve(Y_tilde, nu)
    Y = np.linspace(-6.5, 6.5, 100)
    fig = make_subplots(rows=1, cols=2, subplot_titles=("(a) The Phillips curve", "(b) Inflation, year by year"))
    fig.add_trace(go.Scatter(x=Y, y=phillips_curve(Y, nu), name="Phillips curve", line=dict(color=TEAL, width=4)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=[0, Y_tilde], y=[0, dpi], name="At potential and in the boom", mode="markers",
                             marker=dict(color="black", size=12)), row=1, col=1)
    fig.add_trace(go.Scatter(x=years, y=pi_path, name="Inflation, π", mode="lines+markers",
                             line=dict(color=BLUE, width=3), marker=dict(size=9)), row=1, col=2)
    fig.add_trace(go.Scatter(x=[0, 5], y=[pi_bar, pi_bar], name="Long-run inflation, π̄", mode="lines",
                             line=dict(color=GRAY, dash="dash")), row=1, col=2)
    up = pi_path[-1] >= pi_bar                            # does inflation end above its long-run rate?
    add_end_label(fig, 5, pi_path[-1], "π", BLUE, "bottom" if up else "top", 1, 2)
    add_end_label(fig, 5, pi_bar, "π̄", GRAY, "top" if up else "bottom", 1, 2)
    fig.update_xaxes(title_text="Short-run output, Ỹ (percent)", range=[-6.5, 6.5], tickvals=[-4, -2, 0, 2, 4, 6], row=1, col=1)
    fig.update_yaxes(title_text="Change in inflation, Δπ", range=[-6.5, 6.5], tickvals=[-4, -2, 0, 2, 4, 6], row=1, col=1)
    fig.update_xaxes(title_text="Years of the boom", range=[-0.3, 5.3], row=1, col=2)
    fig.update_yaxes(title_text="Inflation (percent)", range=[-36, 42], tickvals=[-30, -20, -10, 0, 10, 20, 30], row=1, col=2)
    fig.update_layout(title=f"Δπ = {nu:.2f} × ({Y_tilde:+.2f}) = {dpi:+.2f} a year: "
                            f"π = {pi_path[0]:.2f} → {pi_path[1]:.2f} → {pi_path[2]:.2f} → {pi_path[3]:.2f}%",
                      showlegend=False)
    return fig

This time we use `interact_manual`. It adds a button, so nothing happens until you press it. Set the three
sliders, then press **Run the economy**.

In [86]:
run_economy = interact_manual.options(manual_name="Run the economy")
run_economy(booming_economy,
            pi_bar=FloatSlider(value=2.0, min=0.0, max=6.0, step=0.5, description="long-run inflation π̄ (%)", style=WIDE),
            nu=FloatSlider(value=1.0, min=0.1, max=1.5, step=0.01, description="slope ν", style=WIDE),
            Y_tilde=FloatSlider(value=3.0, min=-4.0, max=4.0, step=0.5, description="short-run output Ỹ (%)", style=WIDE));

In [87]:
# @title Website version: Fig. 9 with a precomputed slider for the slope of the Phillips curve
slider_figure(booming_economy, "nu", [0.25, 0.33, 0.5, 0.75, 1.0, 1.5], start=1.0, label="slope of the Phillips curve ν")

**Source.** Adapted from Fig. 9.9 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612),
the textbook's worked answer to Exercise 3 of Chapter 9 ("Overstimulating the economy"), which also inspired panel (b):
the simulated inflation path is our own addition.

**What to notice.**

- Holding output above potential does not raise inflation once: it raises it **every year**. The cost of a
  permanent boom is ever-rising inflation.
- The steeper the Phillips curve, the faster inflation climbs. With the U.S. slope of about 0.33, inflation rises by
  about 1 point a year instead of 3, slower but without end.
- Set $\tilde{Y}$ below zero and the same mechanism brings inflation *down*: that is how a recession fights inflation.

**Your turn.** The textbook's oil-shock exercise starts with inflation at 6 percent. With a slope of $\bar{\nu} = 0.5$,
what constant short-run output brings inflation down to 3 percent in exactly 3 years? Find it with the sliders,
then check it with `simulate_inflation`.

<details><summary>Hint</summary>

Inflation must fall by 3 points in 3 years, so by 1 point a year. Solve $-1 = 0.5 \times \tilde{Y}$, then call
`simulate_inflation(pi_bar=6.0, nu=0.5, Y_tilde=...)`.

</details>

In [88]:
# Your turn: try it here

## Concluding remarks

- Actual output is potential output times a cycle factor, $Y_t = \bar{Y}_t(1+\tilde{Y}_t)$, equations (1) and (2). **Short-run output**
  $\tilde{Y}_t = (Y_t - \bar{Y}_t)/\bar{Y}_t$ is positive in a boom and negative in a recession; it is close to the log difference
  $\ln Y_t - \ln\bar{Y}_t$ (equation 3), and actual growth is trend growth plus the change in the gap (equation 4).
- U.S. recessions push output a few percent below potential, cost millions of jobs and, in total, several percent
  of a year's GDP. Potential output is only estimated, so the measured gaps change when the data are revised.
- Inflation usually falls during a recession. The **Phillips curve** $\Delta\pi = \bar{\nu}\tilde{Y}$, equation (5), captures this, with a
  U.S. slope of about 1/3. Solved forward, inflation is a running total of past gaps (equation 6): a lasting boom
  raises it without end, and lowering it costs a cumulative slump of $1/\bar{\nu}$ percent-years per point (equation 7).
- **Okun's law**, equation (8), links output and unemployment: each point of output below potential adds about half a point
  of unemployment, and unemployment rises whenever output grows more slowly than potential (equation 9).
- The next chapter uses this short-run lens on the Great Recession and the Covid-19 pandemic.